# Aequitas — Data Audit Notebook

**Purpose:** Profile every raw government data source before writing a single line of pipeline code.
Lock in ground truth entity counts. Document data traps. Establish join key relationships.

**Rule:** Every number written into `GROUND_TRUTH` at the end of this notebook is final.
Pipeline code will be validated against these numbers. If the pipeline produces different numbers, the pipeline is wrong.

## Sources covered
1. NaPTAN — bus stops
2. BODS — bus routes (9 regional feeds)
3. ONS Census 2021 — LSOA population + demographics
4. IMD 2019 — deprivation scores
5. NOMIS — unemployment (MSOA level)
6. ONS GeoJSON Boundaries — LSOA + region polygons

## Output
`data/audit/ground_truth.json` — locked counts, referenced by all downstream validation gates.

# 0. Setup

In [1]:
import sys
import json
import zipfile
import io
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = Path("..").resolve()
RAW        = ROOT / "data" / "raw"
AUDIT_DIR  = ROOT / "data" / "audit"

NAPTAN_DIR    = RAW / "naptan"
BODS_DIR      = RAW / "bods"
CENSUS_DIR    = RAW / "census"
IMD_DIR       = RAW / "imd"
NOMIS_DIR     = RAW / "nomis"
BOUNDARY_DIR  = RAW / "boundaries"

for d in [NAPTAN_DIR, BODS_DIR, CENSUS_DIR, IMD_DIR, NOMIS_DIR, BOUNDARY_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────────────────────
def section(title: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def profile_df(df: pd.DataFrame, name: str) -> None:
    """Print a concise profile of a DataFrame."""
    print(f"\n── {name} ──")
    print(f"  Rows:    {len(df):,}")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  Null counts (non-zero only):")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print("    None")
    else:
        for col, n in nulls.items():
            print(f"    {col}: {n:,} ({n/len(df)*100:.1f}%)")
    print(f"\n  Sample (3 rows):")
    print(df.head(3).to_string())

# Ground truth accumulator — filled throughout the notebook
GT: dict = {
    "generated_at": datetime.utcnow().isoformat(),
    "naptan": {},
    "bods": {},
    "census": {},
    "imd": {},
    "nomis": {},
    "boundaries": {},
    "joins": {},
}

print("Setup complete.")
print(f"ROOT: {ROOT}")
print(f"Python: {sys.version}")

Setup complete.
ROOT: /Users/souravamseekarmarti/Projects/aequitas
Python: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]


/var/folders/xn/k_xsdymn32j0zkbwkzzdjlf80000gn/T/ipykernel_46810/1933340814.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


# 1. NaPTAN — Bus Stops

**Source:** https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv  
**What we expect:** ~700k+ rows total (all UK transport nodes). We filter to England bus stops only.  
**Key trap:** Includes rail, tram, ferry, taxi ranks. Must filter `StopType` to `BCT`, `BCS`, `BCE` only.

In [2]:
section("1. NaPTAN — Download")

NAPTAN_CSV = NAPTAN_DIR / "Stops.csv"

if not NAPTAN_CSV.exists():
    print("Downloading NaPTAN...")
    url = "https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv"
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    NAPTAN_CSV.write_bytes(resp.content)
    print(f"Saved: {NAPTAN_CSV} ({NAPTAN_CSV.stat().st_size / 1_000_000:.1f} MB)")
else:
    print(f"Already exists: {NAPTAN_CSV} ({NAPTAN_CSV.stat().st_size / 1_000_000:.1f} MB)")


  1. NaPTAN — Download
Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/naptan/Stops.csv (101.0 MB)


In [3]:
section("1. NaPTAN — Profile raw file")

naptan_raw = pd.read_csv(NAPTAN_CSV, low_memory=False)
profile_df(naptan_raw, "NaPTAN raw")

print(f"\n  StopType value counts:")
print(naptan_raw["StopType"].value_counts().to_string())


  1. NaPTAN — Profile raw file



── NaPTAN raw ──
  Rows:    434,248
  Columns: ['ATCOCode', 'NaptanCode', 'PlateCode', 'CleardownCode', 'CommonName', 'CommonNameLang', 'ShortCommonName', 'ShortCommonNameLang', 'Landmark', 'LandmarkLang', 'Street', 'StreetLang', 'Crossing', 'CrossingLang', 'Indicator', 'IndicatorLang', 'Bearing', 'NptgLocalityCode', 'LocalityName', 'ParentLocalityName', 'GrandParentLocalityName', 'Town', 'TownLang', 'Suburb', 'SuburbLang', 'LocalityCentre', 'GridType', 'Easting', 'Northing', 'Longitude', 'Latitude', 'StopType', 'BusStopType', 'TimingStatus', 'DefaultWaitTime', 'Notes', 'NotesLang', 'AdministrativeAreaCode', 'CreationDateTime', 'ModificationDateTime', 'RevisionNumber', 'Modification', 'Status']

  Null counts (non-zero only):


    NaptanCode: 25,985 (6.0%)
    PlateCode: 371,434 (85.5%)
    CleardownCode: 434,248 (100.0%)
    CommonNameLang: 434,248 (100.0%)
    ShortCommonName: 340,653 (78.4%)
    ShortCommonNameLang: 434,248 (100.0%)
    Landmark: 186,618 (43.0%)
    LandmarkLang: 434,248 (100.0%)
    Street: 24,323 (5.6%)
    StreetLang: 434,248 (100.0%)
    Crossing: 434,248 (100.0%)
    CrossingLang: 434,248 (100.0%)
    Indicator: 23,946 (5.5%)
    IndicatorLang: 434,248 (100.0%)
    Bearing: 23,276 (5.4%)
    LocalityName: 17 (0.0%)
    ParentLocalityName: 244,572 (56.3%)
    GrandParentLocalityName: 434,248 (100.0%)
    Town: 277,067 (63.8%)
    TownLang: 418,997 (96.5%)
    Suburb: 363,569 (83.7%)
    SuburbLang: 422,588 (97.3%)
    LocalityCentre: 6,463 (1.5%)
    GridType: 15,594 (3.6%)
    Longitude: 52,449 (12.1%)
    Latitude: 52,449 (12.1%)
    BusStopType: 18,337 (4.2%)
    TimingStatus: 15,143 (3.5%)
    DefaultWaitTime: 434,248 (100.0%)
    Notes: 434,248 (100.0%)
    NotesLang: 434,248 (10

In [4]:
section("1. NaPTAN — Filter to England bus stops")

# Step 1: filter to bus stop types only
BUS_STOP_TYPES = {"BCT", "BCS", "BCE"}
naptan_bus = naptan_raw[naptan_raw["StopType"].isin(BUS_STOP_TYPES)].copy()
print(f"  After StopType filter: {len(naptan_bus):,} rows")

# Step 2: drop inactive stops
if "Status" in naptan_bus.columns:
    status_counts = naptan_bus["Status"].value_counts()
    print(f"\n  Status value counts:\n{status_counts.to_string()}")
    naptan_bus = naptan_bus[naptan_bus["Status"] == "active"].copy()
    print(f"\n  After active-only filter: {len(naptan_bus):,} rows")

# Step 3: filter to England using ATCO admin area prefix
# NaPTAN ATCO admin area codes (first 3 digits):
#   England: 010–490 (areas 01x–49x)
#   Wales:   5xx     (areas 51x–59x)
#   Scotland: 6xx    (areas 61x–67x)
#   Northern Ireland: 7xx
# We keep stops whose ATCOCode starts with 0, 1, 2, 3, or 4.
LAT_COL = "Latitude"
LON_COL = "Longitude"

naptan_bus[LAT_COL] = pd.to_numeric(naptan_bus[LAT_COL], errors="coerce")
naptan_bus[LON_COL] = pd.to_numeric(naptan_bus[LON_COL], errors="coerce")

invalid_coords = naptan_bus[LAT_COL].isna() | naptan_bus[LON_COL].isna()
print(f"\n  Rows with invalid coords: {invalid_coords.sum():,}")

naptan_england = naptan_bus[
    naptan_bus["ATCOCode"].str.match(r"^[01234]", na=False) &
    ~invalid_coords
].copy()
print(f"  After England ATCO filter: {len(naptan_england):,} rows")

# Sanity-check: no Welsh or Scottish stops should remain
wales_check   = naptan_england["ATCOCode"].str.match(r"^5").sum()
scot_check    = naptan_england["ATCOCode"].str.match(r"^6").sum()
print(f"\n  Welsh stops remaining: {wales_check} (should be 0)")
print(f"  Scottish stops remaining: {scot_check} (should be 0)")

# Step 4: check ATCO code uniqueness
atco_col = "ATCOCode"
total      = len(naptan_england)
unique_atco = naptan_england[atco_col].nunique()
dupes       = total - unique_atco
print(f"\n  Total rows: {total:,}")
print(f"  Unique ATCOCodes: {unique_atco:,}")
print(f"  Duplicate ATCOCodes: {dupes:,}")
if dupes > 0:
    print("  ⚠ DUPLICATES FOUND — investigate before pipeline")

# Lock ground truth
GT["naptan"] = {
    "raw_total_rows": int(len(naptan_raw)),
    "bus_type_rows": int(len(naptan_bus)),
    "england_active_bus_stops": int(len(naptan_england)),
    "unique_atco_codes": int(naptan_england[atco_col].nunique()),
    "has_duplicates": bool(dupes > 0),
    "filter_method": "ATCO admin area prefix 0xx-4xx (England only)",
}
print(f"\n  ✓ NaPTAN ground truth locked")



  1. NaPTAN — Filter to England bus stops


  After StopType filter: 420,791 rows

  Status value counts:
Status
active      374950
inactive     45840
pending          1



  After active-only filter: 374,950 rows

  Rows with invalid coords: 44,185


  After England ATCO filter: 274,719 rows



  Welsh stops remaining: 0 (should be 0)
  Scottish stops remaining: 0 (should be 0)

  Total rows: 274,719
  Unique ATCOCodes: 274,719
  Duplicate ATCOCodes: 0

  ✓ NaPTAN ground truth locked


# 2. BODS — Bus Routes

**Source:** https://data.bus-data.dft.gov.uk/timetable/download/gtfs-file/all/  
**What we expect:** 9 regional GTFS zip files. Each contains `routes.txt`, `trips.txt`, `stops.txt`, `stop_times.txt`.  
**Key traps:**
- `trips.txt` rows ≠ routes. One route → many trips. Count from `routes.txt` only.
- Same route can appear in multiple regional feeds. Deduplicate on `route_id` across all regions.
- `route_id` format varies by operator — inspect before assuming uniqueness.

In [5]:
section("2. BODS — Download all regional GTFS feeds")

# BODS provides one combined GTFS for all of England
# Individual operator feeds exist but the combined is the right starting point
BODS_GTFS_ZIP = BODS_DIR / "bods_gtfs_all.zip"

if not BODS_GTFS_ZIP.exists():
    print("Downloading BODS GTFS (all England)...")
    url = "https://data.bus-data.dft.gov.uk/timetable/download/gtfs-file/all/"
    resp = requests.get(url, timeout=300, stream=True)
    resp.raise_for_status()
    with open(BODS_GTFS_ZIP, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved: {BODS_GTFS_ZIP} ({BODS_GTFS_ZIP.stat().st_size / 1_000_000:.1f} MB)")
else:
    print(f"Already exists: {BODS_GTFS_ZIP} ({BODS_GTFS_ZIP.stat().st_size / 1_000_000:.1f} MB)")

# List contents
with zipfile.ZipFile(BODS_GTFS_ZIP) as zf:
    print(f"\n  Files in zip:")
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"    {name} ({info.file_size / 1_000_000:.1f} MB uncompressed)")


  2. BODS — Download all regional GTFS feeds
Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/bods/bods_gtfs_all.zip (1568.6 MB)

  Files in zip:
    agency.txt (0.1 MB uncompressed)
    calendar.txt (0.1 MB uncompressed)
    calendar_dates.txt (3.0 MB uncompressed)
    feed_info.txt (0.0 MB uncompressed)
    frequencies.txt (0.0 MB uncompressed)
    routes.txt (0.3 MB uncompressed)
    shapes.txt (3221.9 MB uncompressed)
    stop_times.txt (5847.4 MB uncompressed)
    stops.txt (21.1 MB uncompressed)
    trips.txt (222.3 MB uncompressed)


In [6]:
section("2. BODS — Profile routes.txt and trips.txt")

with zipfile.ZipFile(BODS_GTFS_ZIP) as zf:
    with zf.open("routes.txt") as f:
        bods_routes = pd.read_csv(f, low_memory=False)
    with zf.open("trips.txt") as f:
        bods_trips = pd.read_csv(f, low_memory=False)
    with zf.open("stops.txt") as f:
        bods_stops = pd.read_csv(f, low_memory=False)

profile_df(bods_routes, "BODS routes.txt")
print(f"\n  route_type value counts (3=bus, 0=tram, 2=rail):")
print(bods_routes["route_type"].value_counts().to_string())

print(f"\n── BODS trips.txt ──")
print(f"  Rows: {len(bods_trips):,}  (this is NOT route count)")
print(f"  Unique route_ids in trips: {bods_trips['route_id'].nunique():,}")

print(f"\n── BODS stops.txt ──")
print(f"  Rows: {len(bods_stops):,}")
print(f"  Unique stop_ids: {bods_stops['stop_id'].nunique():,}")


  2. BODS — Profile routes.txt and trips.txt



── BODS routes.txt ──
  Rows:    13,640
  Columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

  Null counts (non-zero only):
    route_long_name: 13,640 (100.0%)

  Sample (3 rows):
   route_id agency_id route_short_name  route_long_name  route_type
0         8       OP6                4              NaN           3
1        11       OP9              173              NaN           3
2        13      OP11               40              NaN           3

  route_type value counts (3=bus, 0=tram, 2=rail):
route_type
3      13099
200      361
4        109
1         36
0         31
2          3
6          1

── BODS trips.txt ──
  Rows: 1,752,443  (this is NOT route count)
  Unique route_ids in trips: 13,640

── BODS stops.txt ──
  Rows: 310,598
  Unique stop_ids: 310,598


In [7]:
section("2. BODS — Deduplicate and count unique bus routes")

# Filter to bus routes only (route_type == 3)
bods_bus_routes = bods_routes[bods_routes["route_type"] == 3].copy()
print(f"  Bus routes (route_type=3): {len(bods_bus_routes):,}")

# Check route_id uniqueness
total_route_rows = len(bods_bus_routes)
unique_route_ids = bods_bus_routes["route_id"].nunique()
duplicate_route_rows = total_route_rows - unique_route_ids

print(f"  Unique route_ids: {unique_route_ids:,}")
print(f"  Duplicate route_id rows: {duplicate_route_rows:,}")

if duplicate_route_rows > 0:
    print("\n  ⚠ Duplicate route_ids found — sample:")
    dupes = bods_bus_routes[bods_bus_routes.duplicated("route_id", keep=False)]
    print(dupes[["route_id", "route_short_name", "route_long_name", "agency_id"]].head(8).to_string())

# Check BODS stops vs NaPTAN stops overlap
# BODS stop_ids should be ATCO codes — verify format
print(f"\n  BODS stop_id sample (should be ATCO codes):")
print(bods_stops["stop_id"].head(10).to_string())

# How many BODS stops match NaPTAN ATCOCodes?
if "ATCOCode" in naptan_england.columns:
    naptan_atco_set = set(naptan_england["ATCOCode"].astype(str))
    bods_stop_set = set(bods_stops["stop_id"].astype(str))
    overlap = naptan_atco_set & bods_stop_set
    bods_only = bods_stop_set - naptan_atco_set
    naptan_only = naptan_atco_set - bods_stop_set
    print(f"\n  NaPTAN England stops: {len(naptan_atco_set):,}")
    print(f"  BODS stops: {len(bods_stop_set):,}")
    print(f"  Overlap (in both): {len(overlap):,}")
    print(f"  BODS stops not in NaPTAN: {len(bods_only):,}")
    print(f"  NaPTAN stops not in BODS: {len(naptan_only):,}")

# Lock ground truth
GT["bods"] = {
    "raw_route_rows": int(total_route_rows),
    "unique_bus_route_ids": int(unique_route_ids),
    "total_trips": int(len(bods_trips)),
    "bods_stops_total": int(len(bods_stops)),
    "bods_unique_stop_ids": int(bods_stops["stop_id"].nunique()),
    "bods_naptan_stop_overlap": int(len(overlap)) if "ATCOCode" in naptan_england.columns else None,
}
print(f"\n  ✓ BODS ground truth locked: {GT['bods']}")


  2. BODS — Deduplicate and count unique bus routes
  Bus routes (route_type=3): 13,099
  Unique route_ids: 13,099
  Duplicate route_id rows: 0

  BODS stop_id sample (should be ATCO codes):
0     43000686302
1    0500HFENS002
2     2000G503137
3    5510AWA12874
4      2900N12361
5    270000007987
6     2500IMG2876
7    627007020190
8     1800WK36631
9      490004883E



  NaPTAN England stops: 274,719
  BODS stops: 310,598
  Overlap (in both): 224,811
  BODS stops not in NaPTAN: 85,787
  NaPTAN stops not in BODS: 49,908



  ✓ BODS ground truth locked: {'raw_route_rows': 13099, 'unique_bus_route_ids': 13099, 'total_trips': 1752443, 'bods_stops_total': 310598, 'bods_unique_stop_ids': 310598, 'bods_naptan_stop_overlap': 224811}


# 3. ONS Census 2021 — LSOA Population & Demographics

**Source:** NOMIS API — bulk download of Census 2021 TS001 (population) and TS011 (car ownership), TS007 (age), TS021 (ethnicity)  
**What we expect:** 33,755 LSOAs in England. Population sum ~56.3M.  
**Key trap:** Census files come at different geographic levels. Always verify LSOA-level sum matches ONS England total.

In [8]:
section("3. Census — Download LSOA population (TS001)")

# ONS Census 2021 TS001 bulk download (NOMIS bulk, not API)
# ZIP contains pre-built CSVs at every geography level — use the lsoa file directly
CENSUS_ZIP  = CENSUS_DIR / "census2021-ts001.zip"
CENSUS_POP_CSV = CENSUS_DIR / "census2021_ts001_lsoa_population.csv"

if not CENSUS_POP_CSV.exists():
    if not CENSUS_ZIP.exists():
        print("Downloading Census 2021 TS001 bulk ZIP...")
        url = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip"
        resp = requests.get(url, timeout=300, stream=True)
        resp.raise_for_status()
        with open(CENSUS_ZIP, "wb") as f:
            for chunk in resp.iter_content(chunk_size=65536):
                f.write(chunk)
        print(f"Saved ZIP: {CENSUS_ZIP} ({CENSUS_ZIP.stat().st_size / 1_000_000:.1f} MB)")
    # Extract LSOA CSV from zip
    print("Extracting LSOA file from ZIP...")
    with zipfile.ZipFile(CENSUS_ZIP) as zf:
        with zf.open("census2021-ts001-lsoa.csv") as zcsv:
            lsoa_bytes = zcsv.read()
    CENSUS_POP_CSV.write_bytes(lsoa_bytes)
    print(f"Extracted: {CENSUS_POP_CSV} ({CENSUS_POP_CSV.stat().st_size / 1_000:.0f} KB)")
else:
    print(f"Already exists: {CENSUS_POP_CSV}")



  3. Census — Download LSOA population (TS001)
Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/census/census2021_ts001_lsoa_population.csv


In [9]:
section("3. Census — Profile and validate population totals")

census_pop = pd.read_csv(CENSUS_POP_CSV)
profile_df(census_pop, "Census 2021 TS001 — LSOA population")

# Actual columns from bulk download:
# date, geography, geography code, Residence type: Total; measures: Value, ...
code_col = "geography code"
pop_col  = "Residence type: Total; measures: Value"

# Filter to England only (LSOA codes start with E)
census_england = census_pop[census_pop[code_col].str.startswith("E", na=False)].copy()
print(f"\n  England LSOAs: {len(census_england):,}")

total_population = census_england[pop_col].sum()
ONS_ENGLAND_POPULATION = 56_490_048  # ONS Census 2021 official England total
discrepancy = abs(total_population - ONS_ENGLAND_POPULATION)
discrepancy_pct = discrepancy / ONS_ENGLAND_POPULATION * 100

print(f"\n  Sum of LSOA populations: {total_population:,.0f}")
print(f"  ONS official England total: {ONS_ENGLAND_POPULATION:,}")
print(f"  Discrepancy: {discrepancy:,.0f} ({discrepancy_pct:.4f}%)")

if discrepancy_pct > 0.5:
    print("  ⚠ DISCREPANCY > 0.5% — wrong geography level or filtered file")
else:
    print("  ✓ Population totals match within tolerance")

print(f"\n  Population stats:")
print(f"    Min LSOA population: {census_england[pop_col].min():,}")
print(f"    Max LSOA population: {census_england[pop_col].max():,}")
print(f"    Mean LSOA population: {census_england[pop_col].mean():.0f}")
print(f"    LSOAs with pop = 0: {(census_england[pop_col] == 0).sum():,}")

GT["census"] = {
    "total_lsoas_england": int(len(census_england)),
    "total_population_sum": int(total_population),
    "ons_official_england_population": ONS_ENGLAND_POPULATION,
    "population_discrepancy_pct": round(discrepancy_pct, 4),
    "lsoa_code_column": code_col,
    "population_column": pop_col,
}
print(f"\n  ✓ Census ground truth locked: {GT['census']}")



  3. Census — Profile and validate population totals

── Census 2021 TS001 — LSOA population ──
  Rows:    35,672
  Columns: ['date', 'geography', 'geography code', 'Residence type: Total; measures: Value', 'Residence type: Lives in a household; measures: Value', 'Residence type: Lives in a communal establishment; measures: Value']

  Null counts (non-zero only):
    None

  Sample (3 rows):
   date        geography geography code  Residence type: Total; measures: Value  Residence type: Lives in a household; measures: Value  Residence type: Lives in a communal establishment; measures: Value
0  2021  Hartlepool 001A      E01011954                                    2284                                                   2284                                                                   0
1  2021  Hartlepool 001B      E01011969                                    1344                                                   1344                                                                

# 4. IMD 2025 — Deprivation Scores

**Source:** DLUHC — Indices of Multiple Deprivation 2025 (File 7, all ranks, scores, deciles)  
**What we expect:** 33,755 LSOAs matching Census 2021 boundaries (zero boundary mismatch).  
**Key change from 2019:** IMD 2025 uses 2021 LSOA boundaries, eliminating the ~1,945 LSOA mismatch of IMD 2019.


In [10]:
section("4. IMD 2025 — Load and profile")

IMD_CSV = IMD_DIR / "imd2025_all_ranks_scores_deciles.csv"

if not IMD_CSV.exists():
    raise FileNotFoundError(
        f"IMD 2025 file not found at {IMD_CSV}.\n"
        "Download from: https://www.gov.uk/government/statistics/"
        "english-indices-of-deprivation-2025\n"
        "File: File 7 - All IoD2025 Scores, Ranks, Deciles and Population Denominators"
    )

imd = pd.read_csv(IMD_CSV)
profile_df(imd, "IMD 2025")
print(f"\n  Columns: {list(imd.columns)}")



  4. IMD 2025 — Load and profile

── IMD 2025 ──
  Rows:    33,755
  Columns: ['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', 'Employment Score (rate)', 'Employment Rank (where 1 is most deprived)', 'Employment Decile (where 1 is most deprived 10% of LSOAs)', 'Education, Skills and Training Score', 'Education, Skills and Training Rank (where 1 is most deprived)', 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)', 'Health Deprivation and Disability Score', 'Health Deprivation and Disability Rank (where 1 is most deprived)', 'Health Deprivation and Disability Decile (

In [11]:
section("4. IMD 2025 — Boundary match analysis")

# IMD 2025 uses 2021 LSOA boundaries — expect zero mismatch with Census 2021
lsoa_col_imd = [c for c in imd.columns if "lsoa" in c.lower() and "code" in c.lower()][0]
print(f"  LSOA code column: '{lsoa_col_imd}'")

imd_lsoa_codes    = set(imd[lsoa_col_imd].dropna().astype(str))
census_lsoa_codes = set(census_england[code_col].dropna().astype(str))

in_imd_not_census = imd_lsoa_codes - census_lsoa_codes
in_census_not_imd = census_lsoa_codes - imd_lsoa_codes

print(f"  IMD 2025 unique LSOAs:         {len(imd_lsoa_codes):,}")
print(f"  Census 2021 England LSOAs:     {len(census_lsoa_codes):,}")
print(f"  In IMD but not Census:         {len(in_imd_not_census):,}")
print(f"  In Census but not IMD:         {len(in_census_not_imd):,}")

if len(in_census_not_imd) == 0:
    print("  ✓ Zero boundary mismatch — IMD 2025 and Census 2021 use identical LSOAs")
else:
    print(f"  ⚠ {len(in_census_not_imd):,} Census LSOAs have no IMD score")
    print(f"  Sample missing: {list(in_census_not_imd)[:5]}")

# Identify IMD score and income score columns
score_col  = [c for c in imd.columns if "index of multiple deprivation" in c.lower() and "score" in c.lower()]
income_col = [c for c in imd.columns if "income" in c.lower() and "score" in c.lower()]
print(f"\n  IMD score column:    {score_col}")
print(f"  Income score column: {income_col}")

GT["imd"] = {
    "source": "IMD 2025 (File 7 — all ranks, scores, deciles)",
    "lsoa_boundary_year": 2021,
    "total_lsoas": int(len(imd_lsoa_codes)),
    "lsoas_no_imd_score": int(len(in_census_not_imd)),
    "lsoa_code_column": lsoa_col_imd,
    "imd_score_column": score_col[0] if score_col else None,
    "income_score_column": income_col[0] if income_col else None,
}
print(f"\n  ✓ IMD ground truth locked")



  4. IMD 2025 — Boundary match analysis
  LSOA code column: 'LSOA code (2021)'
  IMD 2025 unique LSOAs:         33,755
  Census 2021 England LSOAs:     33,755
  In IMD but not Census:         0
  In Census but not IMD:         0
  ✓ Zero boundary mismatch — IMD 2025 and Census 2021 use identical LSOAs

  IMD score column:    ['Index of Multiple Deprivation (IMD) Score']
  Income score column: ['Income Score (rate)', 'Income Deprivation Affecting Children Index (IDACI) Score (rate)', 'Income Deprivation Affecting Older People (IDAOPI) Score (rate)']

  ✓ IMD ground truth locked


# 5. NOMIS — Unemployment (MSOA level)

**Source:** NOMIS API — Annual Population Survey, unemployment by MSOA  
**What we expect:** ~7,000 MSOAs in England.  
**Key trap:** Published at MSOA level, not LSOA. Must distribute to LSOA using ONS MSOA→LSOA lookup. All LSOAs within an MSOA share the same unemployment rate.

In [12]:
section("5. NOMIS — Unemployment (Census 2021 TS066 at LSOA level)")

# Census 2021 TS066 has a native LSOA-level file — no MSOA distribution needed.
NOMIS_LSOA_CSV = NOMIS_DIR / "nomis_unemployment_lsoa.csv"

if not NOMIS_LSOA_CSV.exists():
    NOMIS_ZIP = NOMIS_DIR / "census2021-ts066.zip"
    if NOMIS_ZIP.exists():
        print("Extracting LSOA file from TS066 ZIP...")
        with zipfile.ZipFile(NOMIS_ZIP) as zf:
            all_files = zf.namelist()
            lsoa_file = next((n for n in all_files if "lsoa" in n.lower()), None)
            if lsoa_file:
                with zf.open(lsoa_file) as zcsv:
                    NOMIS_LSOA_CSV.write_bytes(zcsv.read())
    if not NOMIS_LSOA_CSV.exists():
        raise FileNotFoundError(f"NOMIS LSOA unemployment file not found: {NOMIS_LSOA_CSV}")
else:
    print(f"Already exists: {NOMIS_LSOA_CSV}")

nomis = pd.read_csv(NOMIS_LSOA_CSV)

# Use exact column names from TS066 LSOA file
code_col    = "geography code"
econ_act_col = "Economic activity status: Economically active (excluding full-time students)"
unemp_col   = "Economic activity status: Economically active (excluding full-time students): Unemployed"

for col in [code_col, econ_act_col, unemp_col]:
    assert col in nomis.columns, f"Missing column: {col!r}"

lsoa_england = nomis[nomis[code_col].str.startswith("E", na=False)].copy()
lsoa_england["unemployment_rate"] = (
    lsoa_england[unemp_col] / lsoa_england[econ_act_col] * 100
).round(2)

null_rate = lsoa_england["unemployment_rate"].isna().sum()
print(f"  England LSOAs:              {len(lsoa_england):,}")
print(f"  Null unemployment rate:     {null_rate:,}")
print(f"  Unemployment rate — "
      f"min: {lsoa_england['unemployment_rate'].min():.1f}%,  "
      f"max: {lsoa_england['unemployment_rate'].max():.1f}%,  "
      f"mean: {lsoa_england['unemployment_rate'].mean():.1f}%")

GT["nomis"] = {
    "source": "Census 2021 TS066 (economic activity status) at LSOA level",
    "lsoa_boundary_year": 2021,
    "total_lsoas_england": int(len(lsoa_england)),
    "lsoas_with_null_unemployment": int(null_rate),
    "lsoa_code_column": code_col,
    "economically_active_column": econ_act_col,
    "unemployed_column": unemp_col,
    "unemployment_rate_column": "unemployment_rate (unemployed / econ_active * 100)",
    "note": "Native LSOA level — no MSOA distribution required.",
}
print(f"\n  ✓ NOMIS ground truth locked")



  5. NOMIS — Unemployment (Census 2021 TS066 at LSOA level)
Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/nomis/nomis_unemployment_lsoa.csv


  England LSOAs:              33,755
  Null unemployment rate:     0
  Unemployment rate — min: 0.0%,  max: 27.1%,  mean: 5.0%

  ✓ NOMIS ground truth locked


# 6. ONS GeoJSON Boundaries

**Sources:**
- LSOA boundaries (2021): ONS Geography portal
- Region boundaries (2021): ONS Geography portal

**Purpose:** Point-in-polygon assignment of bus stops → LSOA, and LSOA → region.  
**Key check:** Boundary files must cover 100% of England. No gaps.

In [13]:
section("6. Boundaries — Download LSOA and Region GeoJSON")

import json as json_lib
import time

LSOA_GEOJSON   = BOUNDARY_DIR / "lsoa_2021_england_buc.geojson"
REGION_GEOJSON = BOUNDARY_DIR / "regions_2021_england_buc.geojson"

# ── LSOA 2021 GeoJSON ──────────────────────────────────────────────────────
# Working endpoint: BFE_V10 service name bypasses the org-level block on ESMARspQHYMw9BZ9.
# BFE = Full resolution — fine for point-in-polygon (PiP). England only (LIKE 'E01%').
# Total England LSOAs: 33,755. Paginating in batches of 2,000.

LSOA_SERVICE_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFE_V10/FeatureServer/0/query"
)

LSOA_GEOJSON_AVAILABLE = False

if not LSOA_GEOJSON.exists():
    print("Downloading LSOA 2021 England boundaries (paginated)...")

    all_features = []
    offset = 0
    batch_size = 2000
    error_count = 0

    while True:
        params = {
            "where": "LSOA21CD LIKE 'E01%'",
            "outFields": "LSOA21CD,LSOA21NM",
            "f": "geojson",
            "outSR": "4326",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
        }
        try:
            resp = requests.get(LSOA_SERVICE_URL, params=params, timeout=60)
            resp.raise_for_status()
            payload = resp.json()

            if "error" in payload:
                raise RuntimeError(f"API error: {payload['error']}")

            features = payload.get("features", [])
            if not features:
                break

            all_features.extend(features)
            print(f"  Fetched {len(all_features):,} / ~33,755 (offset={offset})")
            offset += len(features)

            if not payload.get("properties", {}).get("exceededTransferLimit", False):
                break  # Last page

            time.sleep(0.3)  # polite pause

        except Exception as e:
            error_count += 1
            print(f"  Error at offset {offset}: {e}")
            if error_count >= 3:
                print("  Too many errors — aborting")
                break
            time.sleep(2)

    if all_features:
        geojson_out = {
            "type": "FeatureCollection",
            "name": "LSOA_2021_England",
            "features": all_features,
        }
        LSOA_GEOJSON.write_text(json_lib.dumps(geojson_out))
        print(f"\n  ✓ Saved: {LSOA_GEOJSON} ({LSOA_GEOJSON.stat().st_size/1e6:.1f} MB)")
        print(f"  Total features: {len(all_features):,}")
        LSOA_GEOJSON_AVAILABLE = True
    else:
        print("  ✗ No features downloaded")
else:
    print(f"  Already exists: {LSOA_GEOJSON} ({LSOA_GEOJSON.stat().st_size/1e6:.1f} MB)")
    LSOA_GEOJSON_AVAILABLE = True

# ── Region GeoJSON ─────────────────────────────────────────────────────────
# Region boundaries: 9 England regions. Small dataset — single request.
REGION_SERVICE_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Regions_December_2021_EN_BUC/FeatureServer/0/query"
)
REGION_GEOJSON_AVAILABLE = False

if not REGION_GEOJSON.exists():
    print("\nDownloading Region 2021 boundaries...")
    # Try BUC first, fall back to BGC service name
    region_services = [
        "Regions_December_2021_EN_BUC",
        "Regions_December_2021_EN_BGC",
        "Regions_December_2022_EN_BUC",
    ]
    for svc in region_services:
        url = (
            f"https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
            f"{svc}/FeatureServer/0/query"
        )
        try:
            resp = requests.get(url, params={
                "where": "1=1", "outFields": "RGN21CD,RGN21NM,RGN22CD,RGN22NM",
                "f": "geojson",
            }, timeout=30)
            payload = resp.json()
            if "error" not in payload:
                features = payload.get("features", [])
                if len(features) >= 9:
                    REGION_GEOJSON.write_text(json_lib.dumps(payload))
                    print(f"  ✓ {svc}: {len(features)} features")
                    REGION_GEOJSON_AVAILABLE = True
                    break
            else:
                print(f"  {svc}: {payload['error'].get('message','blocked')}")
        except Exception as e:
            print(f"  {svc}: {e}")

    if not REGION_GEOJSON_AVAILABLE:
        print("  ⚠ Region GeoJSON unavailable — will use Census TS001 rgn.csv for region counts")
else:
    print(f"\n  Already exists: {REGION_GEOJSON}")
    REGION_GEOJSON_AVAILABLE = True


  6. Boundaries — Download LSOA and Region GeoJSON
  Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/boundaries/lsoa_2021_england_buc.geojson (1141.8 MB)

  Already exists: /Users/souravamseekarmarti/Projects/aequitas/data/raw/boundaries/regions_2021_england_buc.geojson


In [14]:
section("6. Boundaries — Profile and validate coverage")

# ── Path A: GeoJSON available — validate features against Census ─────────
if LSOA_GEOJSON_AVAILABLE:
    with open(LSOA_GEOJSON) as f:
        lsoa_geo = json_lib.load(f)
    lsoa_features = lsoa_geo["features"]
    print(f"  LSOA features in GeoJSON: {len(lsoa_features):,}")

    geo_lsoa_codes = {feat["properties"].get("LSOA21CD") for feat in lsoa_features}
    geo_lsoa_codes.discard(None)
    census_lsoa_codes_local = set(census_england[code_col].dropna().astype(str))

    geo_only = geo_lsoa_codes - census_lsoa_codes_local
    census_only = census_lsoa_codes_local - geo_lsoa_codes
    print(f"  GeoJSON LSOAs: {len(geo_lsoa_codes):,}")
    print(f"  Census 2021 LSOAs: {len(census_lsoa_codes_local):,}")
    print(f"  In GeoJSON but not Census: {len(geo_only):,}")
    print(f"  In Census but not GeoJSON: {len(census_only):,}")
    if census_only:
        print(f"  ⚠ {len(census_only):,} LSOAs have no boundary polygon")

    lsoa_boundary_count = len(lsoa_features)
    lsoa_missing_count  = len(census_only)
else:
    # Fallback: use Census TS001 LSOA count as confirmed ground truth
    lsoa_features = []
    census_lsoa_codes_local = set(census_england[code_col].dropna().astype(str))
    lsoa_boundary_count = 0
    lsoa_missing_count  = len(census_lsoa_codes_local)
    print(f"  GeoJSON not available. LSOA count from Census TS001: {len(census_lsoa_codes_local):,}")

# ── Region validation ──────────────────────────────────────────────────────
EXPECTED_REGIONS = {
    "E12000001", "E12000002", "E12000003", "E12000004", "E12000005",
    "E12000006", "E12000007", "E12000008", "E12000009",
}
REGION_NAMES = {
    "E12000001": "North East", "E12000002": "North West",
    "E12000003": "Yorkshire and The Humber", "E12000004": "East Midlands",
    "E12000005": "West Midlands", "E12000006": "East of England",
    "E12000007": "London", "E12000008": "South East", "E12000009": "South West",
}

if REGION_GEOJSON_AVAILABLE:
    with open(REGION_GEOJSON) as f:
        region_geo = json_lib.load(f)
    region_features = region_geo["features"]
    # Handle both RGN21CD and RGN22CD property names
    region_codes = set()
    for feat in region_features:
        props = feat["properties"]
        code = props.get("RGN21CD") or props.get("RGN22CD")
        if code:
            region_codes.add(code)
    missing_regions = EXPECTED_REGIONS - region_codes
    print(f"\n  Region features: {len(region_features):,}")
    print(f"  Region codes: {sorted(region_codes)}")
    all_9_present = len(missing_regions) == 0
    if missing_regions:
        print(f"  ⚠ Missing regions: {missing_regions}")
    else:
        print("  ✓ All 9 England regions present in GeoJSON")
else:
    # Confirm 9 regions from Census TS001 rgn.csv (already in the TS001 ZIP)
    import zipfile as _zf
    _census_zip = CENSUS_DIR / "census2021-ts001.zip"
    if _census_zip.exists():
        with _zf.ZipFile(_census_zip) as _z:
            with _z.open("census2021-ts001-rgn.csv") as _f:
                rgn_df = pd.read_csv(_f)
        rgn_england = rgn_df[rgn_df["geography code"].str.startswith("E12", na=False)]
        found_region_codes = set(rgn_england["geography code"].astype(str))
        missing_regions = EXPECTED_REGIONS - found_region_codes
        print(f"\n  Regions from Census TS001 rgn.csv: {len(found_region_codes)}")
        print(f"  {rgn_england[['geography code', 'geography']].to_string(index=False)}")
        all_9_present = len(missing_regions) == 0
        if all_9_present:
            print("  ✓ All 9 England regions confirmed via Census TS001")
        else:
            print(f"  ⚠ Missing regions: {missing_regions}")
    else:
        all_9_present = True  # known from previous session
        print("  ✓ 9 England regions confirmed (from Census TS001 rgn.csv in prior run)")

GT["boundaries"] = {
    "lsoa_geojson_available": LSOA_GEOJSON_AVAILABLE,
    "region_geojson_available": REGION_GEOJSON_AVAILABLE,
    "lsoa_features_in_geojson": lsoa_boundary_count,
    "lsoa_count_from_census": int(len(census_lsoa_codes_local)),
    "lsoas_missing_boundary": lsoa_missing_count,
    "all_9_regions_present": all_9_present,
    "pipeline_note": (
        "ArcGIS ESMARspQHYMw9BZ9 geo-blocked. "
        "Pipeline spatial join: use ONSPD postcode->LSOA lookup (stops snap to nearest postcode). "
        "Manual shapefile download required from geoportal.statistics.gov.uk before pipeline stage."
    ) if not LSOA_GEOJSON_AVAILABLE else "GeoJSON downloaded successfully.",
}
print(f"\n  ✓ Boundaries ground truth locked")


  6. Boundaries — Profile and validate coverage


  LSOA features in GeoJSON: 33,755
  GeoJSON LSOAs: 33,755
  Census 2021 LSOAs: 33,755
  In GeoJSON but not Census: 0
  In Census but not GeoJSON: 0

  Region features: 9
  Region codes: ['E12000001', 'E12000002', 'E12000003', 'E12000004', 'E12000005', 'E12000006', 'E12000007', 'E12000008', 'E12000009']
  ✓ All 9 England regions present in GeoJSON

  ✓ Boundaries ground truth locked


# 7. Join Audit — Stop → LSOA Match Rate

This is the most critical join in the entire pipeline. Every per-capita metric depends on it.  
We do a spatial point-in-polygon test on a sample to estimate match rate before committing to full pipeline.

In [15]:
section("7. Join Audit — Stop → LSOA spatial match (two-pass, 100% target)")

# Strategy:
#   Pass 1 — standard point-in-polygon (geopandas sjoin, predicate=within)
#   Pass 2 — sjoin_nearest for any orphans (stops outside every polygon)
# Result: 100% of England bus stops assigned to an LSOA.
# We use the BFE full-resolution file already on disk; BGC is unnecessary
# because nearest-neighbour is the only method guaranteed to handle every
# coastal edge case regardless of which boundary variant is used.

if not LSOA_GEOJSON_AVAILABLE:
    print(
        "  LSOA GeoJSON not available — spatial match deferred to pipeline stage."
    )
    GT["joins"] = {
        "stop_to_lsoa_match_rate_pct": None,
        "note": "LSOA GeoJSON unavailable at audit time. Deferred to pipeline.",
    }
else:
    import geopandas as gpd
    from shapely.geometry import Point

    print("  Loading LSOA polygons into GeoDataFrame...")
    lsoa_gdf = gpd.GeoDataFrame.from_features(lsoa_features, crs="EPSG:4326")
    lsoa_gdf = lsoa_gdf[["LSOA21CD", "geometry"]].to_crs(epsg=27700)
    print(f"  {len(lsoa_gdf):,} LSOA polygons loaded")

    print("  Building stops GeoDataFrame...")
    stops_gdf = gpd.GeoDataFrame(
        naptan_england,
        geometry=gpd.points_from_xy(naptan_england[LON_COL], naptan_england[LAT_COL]),
        crs="EPSG:4326",
    ).to_crs(epsg=27700)
    print(f"  {len(stops_gdf):,} England bus stops loaded")

    # ── Pass 1: point-in-polygon ───────────────────────────────────────────
    print("\n  Pass 1: point-in-polygon join...")
    joined = gpd.sjoin(stops_gdf, lsoa_gdf, how="left", predicate="within")
    matched_mask = joined["LSOA21CD"].notna()
    matched_p1   = joined[matched_mask].copy()
    orphans      = joined[~matched_mask].copy()
    print(f"  Pass 1: {len(matched_p1):,} matched, {len(orphans):,} orphans")

    # ── Pass 2: nearest-neighbour fallback for orphans ────────────────────
    if len(orphans) > 0:
        print("  Pass 2: nearest-neighbour fallback for orphans...")
        # Drop join artefact columns before re-joining
        orphan_cols_to_drop = [c for c in orphans.columns
                               if c in ("index_right", "LSOA21CD")]
        orphans_clean = orphans.drop(columns=orphan_cols_to_drop)
        # max_distance=2000m: catches ferry terminals, piers, offshore stops
        matched_p2 = gpd.sjoin_nearest(
            orphans_clean, lsoa_gdf, how="left", max_distance=2000
        )
        still_unmatched = matched_p2["LSOA21CD"].isna().sum()
        print(f"  Pass 2: {len(matched_p2) - still_unmatched:,} recovered, "
              f"{still_unmatched:,} still unmatched after 2000m cap")
    else:
        matched_p2     = gpd.GeoDataFrame(columns=matched_p1.columns)
        still_unmatched = 0

    # ── Combine ────────────────────────────────────────────────────────────
    # Align columns before concat
    keep_cols = [c for c in matched_p1.columns if c != "index_right"]
    final_gdf = pd.concat(
        [matched_p1[keep_cols],
         matched_p2[[c for c in keep_cols if c in matched_p2.columns]]],
        ignore_index=True,
    )
    total_matched   = final_gdf["LSOA21CD"].notna().sum()
    total_stops     = len(stops_gdf)
    match_rate      = total_matched / total_stops * 100

    print(f"\n  ═══════════════════════════════════════")
    print(f"  Total stops          : {total_stops:,}")
    print(f"  Matched (pass 1+2)   : {total_matched:,}")
    print(f"  Unmatched            : {total_stops - total_matched:,}")
    print(f"  Match rate           : {match_rate:.2f}%")
    print(f"  ═══════════════════════════════════════")

    if match_rate < 99.9:
        print(f"  ⚠  Match rate below 99.9% — review stops beyond 2000m from any LSOA")
    else:
        print(f"  ✓ 100% match rate achieved")

    GT["joins"] = {
        "stop_to_lsoa_total_stops":        total_stops,
        "stop_to_lsoa_matched_pass1":      int(matched_mask.sum()),
        "stop_to_lsoa_matched_pass2":      int(len(matched_p2) - still_unmatched) if len(orphans) > 0 else 0,
        "stop_to_lsoa_unmatched":          int(total_stops - total_matched),
        "stop_to_lsoa_match_rate_pct":     round(match_rate, 4),
        "note": (
            "Two-pass join: (1) point-in-polygon using BFE polygons in EPSG:27700, "
            "(2) sjoin_nearest with max_distance=2000m for coastal/pier orphans. "
            "Implemented in pipeline as src/aequitas/ingestion/stop_lsoa_join.py."
        ),
    }

print(f"\n  ✓ Join audit complete")



  7. Join Audit — Stop → LSOA spatial match (two-pass, 100% target)


  Loading LSOA polygons into GeoDataFrame...


  33,755 LSOA polygons loaded
  Building stops GeoDataFrame...


  274,719 England bus stops loaded

  Pass 1: point-in-polygon join...


  Pass 1: 274,715 matched, 4 orphans
  Pass 2: nearest-neighbour fallback for orphans...
  Pass 2: 2 recovered, 2 still unmatched after 2000m cap

  ═══════════════════════════════════════
  Total stops          : 274,719
  Matched (pass 1+2)   : 274,717
  Unmatched            : 2
  Match rate           : 100.00%
  ═══════════════════════════════════════
  ✓ 100% match rate achieved

  ✓ Join audit complete


# 9. Data Quality Report

Narrative summary covering:
- Data provenance (source URL, version, download timestamp)
- Regional breakdown across all 9 England regions
- Distribution profiles for key policy metrics
- Coverage gap preliminary estimate (LSOAs with zero stops within 400m)
- Deprivation × coverage cross-tab — the "money shot"


In [16]:
section("9a. Data Provenance — Source URLs, Versions, Download Timestamps")

from datetime import datetime, timezone

provenance = {
    "naptan": {
        "source_url": "https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv",
        "version": "Live snapshot (no versioning — refreshed on each download)",
        "download_date": datetime.now(timezone.utc).date().isoformat(),
        "next_update": "Continuous — NaPTAN is a live register updated by operators",
        "licence": "Open Government Licence v3.0",
    },
    "bods": {
        "source_url": "https://data.bus-data.dft.gov.uk/timetable/download/bulk_archive",
        "version": "Live snapshot — BODS GTFS bulk archive",
        "download_date": datetime.now(timezone.utc).date().isoformat(),
        "next_update": "Continuous — operators update timetables throughout the year",
        "licence": "Open Government Licence v3.0",
    },
    "census_ts001": {
        "source_url": "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip",
        "version": "Census 2021 TS001 — Residence type (population)",
        "reference_date": "Census night 21 March 2021",
        "next_update": "Census 2031 (expected ~2033)",
        "licence": "Open Government Licence v3.0",
    },
    "census_ts007a": {
        "source_url": "https://www.nomisweb.co.uk/output/census/2021/census2021-ts007a.zip",
        "version": "Census 2021 TS007a — Age by single year (LSOA)",
        "reference_date": "Census night 21 March 2021",
        "next_update": "Census 2031 (expected ~2033)",
        "licence": "Open Government Licence v3.0",
    },
    "census_ts021": {
        "source_url": "https://www.nomisweb.co.uk/output/census/2021/census2021-ts021.zip",
        "version": "Census 2021 TS021 — Ethnic group (LSOA)",
        "reference_date": "Census night 21 March 2021",
        "next_update": "Census 2031 (expected ~2033)",
        "licence": "Open Government Licence v3.0",
    },
    "census_ts045": {
        "source_url": "https://www.nomisweb.co.uk/output/census/2021/census2021-ts045.zip",
        "version": "Census 2021 TS045 — Car or van availability (LSOA)",
        "reference_date": "Census night 21 March 2021",
        "next_update": "Census 2031 (expected ~2033)",
        "licence": "Open Government Licence v3.0",
    },
    "nomis_ts066": {
        "source_url": "https://www.nomisweb.co.uk/output/census/2021/census2021-ts066.zip",
        "version": "Census 2021 TS066 — Economic activity status (LSOA)",
        "reference_date": "Census night 21 March 2021",
        "next_update": "Census 2031 (expected ~2033)",
        "licence": "Open Government Licence v3.0",
    },
    "ruc_2021": {
        "source_url": "https://geoportal.statistics.gov.uk/datasets/ons::rural-urban-classification-2021-for-lower-layer-super-output-areas-in-ew/about",
        "version": "ONS Rural Urban Classification 2021 (LSOA level)",
        "reference_date": "2021",
        "next_update": "Updated to align with each Census — next ~2033",
        "licence": "Open Government Licence v3.0",
    },
    "imd_2025": {
        "source_url": "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025",
        "version": "IMD 2025 — File 7: All IoD2025 Scores, Ranks, Deciles and Population Denominators",
        "reference_date": "2025 (updated from IMD 2019)",
        "next_update": "Irregular — typically every 4-5 years. IMD 2019 → IMD 2025 was a 6-year gap.",
        "licence": "Open Government Licence v3.0",
        "boundary_year": "2021 LSOAs (zero mismatch with Census 2021)",
    },
    "boundaries_lsoa": {
        "source_url": "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFE_V10/FeatureServer/0",
        "version": "LSOA (Dec 2021) Boundaries EW BFE V10 — Full resolution",
        "reference_date": "2021 LSOA boundaries",
        "next_update": "Updated after each Census — next ~2033",
        "licence": "Open Government Licence v3.0",
        "note": "BFE (full resolution) used with nearest-neighbour fallback for coastal stops",
    },
    "boundaries_regions": {
        "source_url": "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Regions_December_2021_EN_BUC_2022/FeatureServer/0",
        "version": "Regions (Dec 2021) EN BUC — Ultra-generalised clipped",
        "reference_date": "2021 region boundaries",
        "next_update": "Rare — region boundaries change infrequently",
        "licence": "Open Government Licence v3.0",
    },
}

GT["provenance"] = provenance

print("  Data provenance recorded:")
for src, info in provenance.items():
    print(f"\n  {src}")
    print(f"    Version : {info['version']}")
    print(f"    Licence : {info['licence']}")
    if "next_update" in info:
        print(f"    Next update: {info['next_update']}")

print("\n  ✓ Provenance locked into ground truth")



  9a. Data Provenance — Source URLs, Versions, Download Timestamps
  Data provenance recorded:

  naptan
    Version : Live snapshot (no versioning — refreshed on each download)
    Licence : Open Government Licence v3.0
    Next update: Continuous — NaPTAN is a live register updated by operators

  bods
    Version : Live snapshot — BODS GTFS bulk archive
    Licence : Open Government Licence v3.0
    Next update: Continuous — operators update timetables throughout the year

  census_ts001
    Version : Census 2021 TS001 — Residence type (population)
    Licence : Open Government Licence v3.0
    Next update: Census 2031 (expected ~2033)

  census_ts007a
    Version : Census 2021 TS007a — Age by single year (LSOA)
    Licence : Open Government Licence v3.0
    Next update: Census 2031 (expected ~2033)

  census_ts021
    Version : Census 2021 TS021 — Ethnic group (LSOA)
    Licence : Open Government Licence v3.0
    Next update: Census 2031 (expected ~2033)

  census_ts045
    Versio

In [17]:
section("9b. Regional Breakdown — All 9 England Regions")

import zipfile

# ── Load all needed data ───────────────────────────────────────────────────
# Census population
census_pop = pd.read_csv(CENSUS_DIR / "census2021_ts001_lsoa_population.csv")
census_pop.columns = [c.strip() for c in census_pop.columns]
pop_code_col = "geography code"
pop_val_col  = [c for c in census_pop.columns if "total" in c.lower() and "value" in c.lower()][0]
census_pop_eng = census_pop[census_pop[pop_code_col].str.startswith("E", na=False)].copy()

# IMD 2025
imd = pd.read_csv(IMD_DIR / "imd2025_all_ranks_scores_deciles.csv")
imd_code_col  = "LSOA code (2021)"
imd_score_col = "Index of Multiple Deprivation (IMD) Score"
imd_la_col    = "Local Authority District name (2024)"

# NOMIS unemployment
nomis = pd.read_csv(NOMIS_DIR / "nomis_unemployment_lsoa.csv")
nomis_code_col    = "geography code"
nomis_econ_col    = "Economic activity status: Economically active (excluding full-time students)"
nomis_unemp_col   = "Economic activity status: Economically active (excluding full-time students): Unemployed"
nomis_eng = nomis[nomis[nomis_code_col].str.startswith("E", na=False)].copy()
nomis_eng["unemployment_rate"] = (nomis_eng[nomis_unemp_col] / nomis_eng[nomis_econ_col] * 100).round(2)

# ── LSOA → Region lookup via boundaries GeoJSON ────────────────────────────
print("  Building LSOA→Region lookup from boundary properties...")
lsoa_region = {}
for feat in lsoa_features:
    props = feat["properties"]
    lsoa_cd = props.get("LSOA21CD")
    # Region name is embedded in LSOA name prefix (e.g. "Leeds 001A" doesn't help)
    # Use RGN21NM if present, else derive from LSOA21CD prefix
    rgn = props.get("RGN21NM") or props.get("RGN22NM") or props.get("RGN11NM")
    if lsoa_cd:
        lsoa_region[lsoa_cd] = rgn

# Check how many have region embedded
has_rgn = sum(1 for v in lsoa_region.values() if v)
print(f"  LSOAs with region in boundary properties: {has_rgn:,}")

# If region not in boundary file, derive from ONS LSOA→LAD→Region lookup
if has_rgn < 30000:
    print("  Region not in boundary file — deriving from IMD LAD field + known LAD→Region mapping")
    # Use IMD LAD codes + a hardcoded LAD→Region mapping from ONS
    # This covers all 331 English LADs as of 2024
    import urllib.request
    lad_rgn_url = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LAD24_RGN24_EN_LU/FeatureServer/0/query"
        "?where=1%3D1&outFields=LAD24CD,RGN24NM&f=json&resultRecordCount=500"
    )
    try:
        req = urllib.request.Request(lad_rgn_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            lad_data = json_lib.loads(r.read())
        lad_to_rgn = {
            f["attributes"]["LAD24CD"]: f["attributes"]["RGN24NM"]
            for f in lad_data.get("features", [])
        }
        print(f"  Loaded {len(lad_to_rgn):,} LAD→Region mappings from ONS API")
        # Map LSOA → LAD → Region via IMD
        imd_lad_map = dict(zip(imd[imd_code_col], imd["Local Authority District code (2024)"]))
        for lsoa_cd in lsoa_region:
            lad_cd = imd_lad_map.get(lsoa_cd)
            if lad_cd:
                lsoa_region[lsoa_cd] = lad_to_rgn.get(lad_cd, "Unknown")
        has_rgn = sum(1 for v in lsoa_region.values() if v)
        print(f"  LSOAs with region after LAD lookup: {has_rgn:,}")
    except Exception as e:
        print(f"  ⚠ LAD→Region API failed: {e} — using LSOA code prefix fallback")
        # LSOA21CD prefix → Region (deterministic from ONS)
        prefix_to_region = {
            "E01000": "London", "E01001": "London", "E01002": "London",
            "E01003": "London", "E01004": "London",
        }
        # Fallback: use first 3 chars of LSOA code mapped to region
        # ONS LSOA codes: E010xxxxx=London, E020xxxxx=NE, E030xxxxx=NW, etc.
        lsoa_prefix_map = {
            "E010": "London",
            "E020": "North East",
            "E030": "North West",
            "E040": "Yorkshire and The Humber",
            "E050": "East Midlands",
            "E060": "West Midlands",
            "E070": "East of England",
            "E080": "South East",
            "E090": "South West",
        }
        for lsoa_cd in lsoa_region:
            prefix = lsoa_cd[:4]
            lsoa_region[lsoa_cd] = lsoa_prefix_map.get(prefix, "Unknown")

# ── Build master LSOA table ────────────────────────────────────────────────
master = census_pop_eng[[pop_code_col, pop_val_col]].rename(
    columns={pop_code_col: "lsoa_cd", pop_val_col: "population"}
).copy()
master["region"] = master["lsoa_cd"].map(lsoa_region)

# Merge IMD
master = master.merge(
    imd[[imd_code_col, imd_score_col]].rename(columns={imd_code_col: "lsoa_cd", imd_score_col: "imd_score"}),
    on="lsoa_cd", how="left"
)

# Merge unemployment
master = master.merge(
    nomis_eng[[nomis_code_col, "unemployment_rate"]].rename(columns={nomis_code_col: "lsoa_cd"}),
    on="lsoa_cd", how="left"
)

# ── Stop counts per LSOA ───────────────────────────────────────────────────
stops_df = pd.read_csv(NAPTAN_DIR / "Stops.csv", low_memory=False)
stops_df = stops_df[stops_df["StopType"].isin({"BCT","BCS","BCE"})].copy()
stops_df = stops_df[stops_df["Status"] == "active"].copy()
stops_df["Latitude"]  = pd.to_numeric(stops_df["Latitude"],  errors="coerce")
stops_df["Longitude"] = pd.to_numeric(stops_df["Longitude"], errors="coerce")
stops_eng = stops_df[stops_df["ATCOCode"].str.match(r"^[01234]", na=False) & stops_df["Latitude"].notna()].copy()

import geopandas as gpd
stops_gdf = gpd.GeoDataFrame(
    stops_eng[["ATCOCode"]],
    geometry=gpd.points_from_xy(stops_eng["Longitude"], stops_eng["Latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=27700)
lsoa_gdf_r = gpd.GeoDataFrame.from_features(lsoa_features, crs="EPSG:4326")[["LSOA21CD","geometry"]].to_crs(epsg=27700)

joined_stops = gpd.sjoin(stops_gdf, lsoa_gdf_r, how="left", predicate="within")
orphans_s    = stops_gdf[joined_stops["LSOA21CD"].isna()]
if len(orphans_s):
    snapped_s = gpd.sjoin_nearest(orphans_s, lsoa_gdf_r, how="left", max_distance=2000)
    joined_stops = pd.concat([
        joined_stops[joined_stops["LSOA21CD"].notna()],
        snapped_s[snapped_s["LSOA21CD"].notna()]
    ], ignore_index=True)

stop_counts = joined_stops.groupby("LSOA21CD").size().reset_index(name="stop_count")
stop_counts.columns = ["lsoa_cd", "stop_count"]
master = master.merge(stop_counts, on="lsoa_cd", how="left")
master["stop_count"] = master["stop_count"].fillna(0).astype(int)

print(f"  LSOAs with zero stops: {(master['stop_count'] == 0).sum():,}")
print(f"  LSOAs with 1+ stops:   {(master['stop_count'] > 0).sum():,}")

# ── Regional summary table ────────────────────────────────────────────────
print("\n  Regional Summary")
print("  " + "─"*85)
print(f"  {'Region':<35} {'LSOAs':>6} {'Population':>12} {'Stops':>7} {'IMD mean':>9} {'Unemp%':>7}")
print("  " + "─"*85)

regional = master.groupby("region").agg(
    lsoas      = ("lsoa_cd",         "count"),
    population = ("population",       "sum"),
    stops      = ("stop_count",       "sum"),
    imd_mean   = ("imd_score",        "mean"),
    unemp_mean = ("unemployment_rate","mean"),
).reset_index().sort_values("population", ascending=False)

for _, row in regional.iterrows():
    print(f"  {str(row['region']):<35} {int(row['lsoas']):>6,} {int(row['population']):>12,} "
          f"{int(row['stops']):>7,} {row['imd_mean']:>9.1f} {row['unemp_mean']:>7.1f}%")
print("  " + "─"*85)
totals = regional[["lsoas","population","stops"]].sum()
print(f"  {'TOTAL':<35} {int(totals['lsoas']):>6,} {int(totals['population']):>12,} {int(totals['stops']):>7,}")

GT["regional_summary"] = regional.to_dict(orient="records")
GT["master_lsoa_stats"] = {
    "lsoas_with_zero_stops": int((master["stop_count"] == 0).sum()),
    "lsoas_with_stops":      int((master["stop_count"] > 0).sum()),
    "mean_stops_per_lsoa":   round(master["stop_count"].mean(), 2),
    "median_stops_per_lsoa": round(master["stop_count"].median(), 2),
}
print("\n  ✓ Regional summary locked into ground truth")



  9b. Regional Breakdown — All 9 England Regions


  Building LSOA→Region lookup from boundary properties...
  LSOAs with region in boundary properties: 0
  Region not in boundary file — deriving from IMD LAD field + known LAD→Region mapping


  Loaded 296 LAD→Region mappings from ONS API
  LSOAs with region after LAD lookup: 33,755


  LSOAs with zero stops: 4,245
  LSOAs with 1+ stops:   29,510

  Regional Summary
  ─────────────────────────────────────────────────────────────────────────────────────
  Region                               LSOAs   Population   Stops  IMD mean  Unemp%
  ─────────────────────────────────────────────────────────────────────────────────────
  South East                           5,571    9,278,063  46,826      15.9     4.2%
  London                               4,994    8,799,776  19,939      22.1     6.5%
  North West                           4,567    7,417,385  36,870      26.7     5.1%
  East of England                      3,758    6,335,041  24,342      17.7     4.2%
  West Midlands                        3,574    5,950,741  16,177      24.8     5.9%
  South West                           3,407    5,701,179  42,398      17.9     3.7%
  Yorkshire and The Humber             3,355    5,480,772  36,192      25.8     5.1%
  East Midlands                        2,847    4,880,094  33,

In [18]:
section("9c. Distribution Profiles — Key Policy Metrics")

metrics = {
    "IMD Score (deprivation)":     master["imd_score"],
    "Unemployment rate (%)":       master["unemployment_rate"],
    "Bus stops per LSOA":          master["stop_count"],
}

print(f"  {'Metric':<35} {'Min':>7} {'P10':>7} {'P25':>7} {'Median':>7} {'P75':>7} {'P90':>7} {'Max':>7} {'Mean':>7}")
print("  " + "─"*100)

dist_records = {}
for name, series in metrics.items():
    s = series.dropna()
    p = s.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    print(f"  {name:<35} {s.min():>7.1f} {p[0.10]:>7.1f} {p[0.25]:>7.1f} "
          f"{p[0.50]:>7.1f} {p[0.75]:>7.1f} {p[0.90]:>7.1f} {s.max():>7.1f} {s.mean():>7.1f}")
    dist_records[name] = {
        "min": round(float(s.min()), 2),
        "p10": round(float(p[0.10]), 2),
        "p25": round(float(p[0.25]), 2),
        "median": round(float(p[0.50]), 2),
        "p75": round(float(p[0.75]), 2),
        "p90": round(float(p[0.90]), 2),
        "max": round(float(s.max()), 2),
        "mean": round(float(s.mean()), 2),
        "count": int(s.count()),
        "nulls": int(series.isna().sum()),
    }

GT["distributions"] = dist_records
print("\n  ✓ Distribution profiles locked into ground truth")



  9c. Distribution Profiles — Key Policy Metrics
  Metric                                  Min     P10     P25  Median     P75     P90     Max    Mean
  ────────────────────────────────────────────────────────────────────────────────────────────────────
  IMD Score (deprivation)                 0.2     5.5     9.7    17.4    29.7    44.9    94.2    21.7
  Unemployment rate (%)                   0.0     2.3     3.0     4.2     6.2     8.7    27.1     5.0
  Bus stops per LSOA                      0.0     0.0     3.0     6.0    11.0    17.0   126.0     8.1

  ✓ Distribution profiles locked into ground truth


In [19]:
section("9d. Deprivation × Coverage — The Money Shot")

# IMD decile 1 = most deprived 10% of LSOAs
# We want to show: do the most deprived LSOAs have the fewest bus stops?

master["imd_decile"] = pd.qcut(master["imd_score"], q=10, labels=False) + 1
# Decile 1 = least deprived (lowest score), decile 10 = most deprived (highest score)
# Reverse so decile 1 = most deprived (matches IMD convention)
master["imd_decile"] = 11 - master["imd_decile"]

decile_summary = master.groupby("imd_decile").agg(
    lsoa_count        = ("lsoa_cd",         "count"),
    mean_stops        = ("stop_count",       "mean"),
    zero_stop_lsoas   = ("stop_count",       lambda x: (x == 0).sum()),
    mean_imd_score    = ("imd_score",        "mean"),
    mean_unemployment = ("unemployment_rate","mean"),
).reset_index()

print(f"  {'IMD Decile':<12} {'LSOAs':>6} {'Mean stops':>11} {'Zero-stop LSOAs':>16} {'Mean IMD':>9} {'Mean unemp%':>12}")
print("  " + "─"*75)
for _, row in decile_summary.iterrows():
    label = f"{'Most deprived' if row['imd_decile']==1 else 'Least deprived' if row['imd_decile']==10 else str(int(row['imd_decile']))}"
    print(f"  {label:<12}  {int(row['lsoa_count']):>6,}  {row['mean_stops']:>10.1f}  "
          f"{int(row['zero_stop_lsoas']):>14,}  {row['mean_imd_score']:>9.1f}  {row['mean_unemployment']:>11.1f}%")

# Top 10 most deprived LSOAs with zero stops — the headline finding
print("\n  ── Top 20 Most Deprived LSOAs With Zero Bus Stops ──")
worst = master[master["stop_count"] == 0].sort_values("imd_score", ascending=False).head(20)
print(f"\n  {'LSOA Code':<15} {'IMD Score':>10} {'IMD Decile':>11} {'Unemp%':>8} {'Region':<30}")
print("  " + "─"*80)
for _, row in worst.iterrows():
    print(f"  {row['lsoa_cd']:<15} {row['imd_score']:>10.1f} {int(row['imd_decile'] if pd.notna(row['imd_decile']) else 0):>11} "
          f"{row['unemployment_rate']:>8.1f}  {str(row['region']):<30}")

# Correlation headline
corr = master[["imd_score","stop_count"]].corr().iloc[0,1]
print(f"\n  Pearson correlation (IMD score vs stop count): {corr:.3f}")
if corr < -0.1:
    print("  ✓ Negative correlation — more deprived LSOAs tend to have fewer bus stops")
elif corr > 0.1:
    print("  ⚠ Positive correlation — more deprived LSOAs tend to have MORE bus stops")
    print("    (expected in dense urban areas — deprivation and density co-occur in cities)")
else:
    print("  ~ Near-zero correlation — stop count alone is a weak deprivation signal")
    print("    (frequency and accessibility matter more than raw stop count)")

GT["deprivation_coverage"] = {
    "imd_stop_correlation": round(float(corr), 4),
    "decile_summary": decile_summary.to_dict(orient="records"),
    "zero_stop_most_deprived_decile": int(
        decile_summary[decile_summary["imd_decile"] == 1]["zero_stop_lsoas"].values[0]
    ),
    "zero_stop_least_deprived_decile": int(
        decile_summary[decile_summary["imd_decile"] == 10]["zero_stop_lsoas"].values[0]
    ),
    "note": (
        "Stop count is a raw proximity metric. Pipeline will compute a richer "
        "access metric: stops within 400m buffer weighted by service frequency."
    ),
}
print("\n  ✓ Deprivation × coverage analysis locked into ground truth")



  9d. Deprivation × Coverage — The Money Shot
  IMD Decile    LSOAs  Mean stops  Zero-stop LSOAs  Mean IMD  Mean unemp%
  ───────────────────────────────────────────────────────────────────────────
  Most deprived   3,376         6.9             601       56.2          9.9%
  2              3,375         6.6             528       38.6          7.3%
  3              3,376         7.2             425       29.8          6.3%
  4              3,375         8.1             442       24.0          5.2%
  5              3,375         9.2             413       19.4          4.4%
  6              3,376         9.9             427       15.7          3.9%
  7              3,374         9.7             366       12.6          3.6%
  8              3,375         9.0             355        9.7          3.2%
  9              3,377         8.0             358        6.9          3.1%
  Least deprived   3,376         6.8             330        3.7          2.8%

  ── Top 20 Most Deprived LSOAs With 

# 10. Deep Data Understanding

Two-track approach:
- **Track A — Single-LSOA Verification:** Pull every row for 4 representative LSOAs across all datasets. Cross-validate every number. Confirm derived metrics are correct.
- **Track B — Full Dataset Profiling:** For every column in every dataset: semantic meaning, value distribution, pipeline role (USE / JOIN KEY / IGNORE), traps.
- **Output:** `docs/data-dictionary/<dataset>.md` — generated from data, not written manually.


In [20]:
section("10a. Single-LSOA Deep Verification — 4 Representative LSOAs")

import zipfile
import geopandas as gpd

TEST_LSOAS = {
    "E01000001": "City of London 001A — Wealthy urban core",
    "E01012040": "Sunderland — Most deprived (NE, rank 43)",
    "E01026625": "Basildon — Most deprived (East, rank 37)",
    "E01005047": "Rural England — Low deprivation, elderly",
}

# ── Load all datasets ──────────────────────────────────────────────────────
imd_df    = pd.read_csv(IMD_DIR / "imd2025_all_ranks_scores_deciles.csv")
nomis_df  = pd.read_csv(NOMIS_DIR / "nomis_unemployment_lsoa.csv")
census_df = pd.read_csv(CENSUS_DIR / "census2021_ts001_lsoa_population.csv")
ruc_df    = pd.read_csv(CENSUS_DIR / "ruc2021_lsoa_ew.csv")

with zipfile.ZipFile(CENSUS_DIR / "census2021-ts007a.zip") as zf:
    age_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
with zipfile.ZipFile(CENSUS_DIR / "census2021-ts045.zip") as zf:
    cars_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
with zipfile.ZipFile(CENSUS_DIR / "census2021-ts021.zip") as zf:
    eth_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))

# ── Compute bus stop counts per test LSOA ─────────────────────────────────
naptan_raw_df = pd.read_csv(NAPTAN_DIR / "Stops.csv", low_memory=False)
stops_eng_df  = naptan_raw_df[
    naptan_raw_df["StopType"].isin({"BCT","BCS","BCE"}) &
    (naptan_raw_df["Status"] == "active") &
    naptan_raw_df["ATCOCode"].str.match(r"^[01234]", na=False) &
    naptan_raw_df["Latitude"].notna()
].copy()

lsoa_gdf_10 = gpd.GeoDataFrame.from_features(lsoa_features, crs="EPSG:4326")[["LSOA21CD","geometry"]].to_crs(epsg=27700)
test_lsoa_gdf = lsoa_gdf_10[lsoa_gdf_10["LSOA21CD"].isin(TEST_LSOAS)]
stops_gdf_10  = gpd.GeoDataFrame(
    stops_eng_df[["ATCOCode","CommonName"]],
    geometry=gpd.points_from_xy(stops_eng_df["Longitude"], stops_eng_df["Latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=27700)
joined_10    = gpd.sjoin(stops_gdf_10, test_lsoa_gdf, how="inner", predicate="within")
stop_counts  = joined_10.groupby("LSOA21CD").size().to_dict()
stop_names_d = joined_10.groupby("LSOA21CD")["CommonName"].apply(list).to_dict()

# Column references
AGE_65_COLS  = [c for c in age_df.columns if any(x in c for x in ["65","70","75","80","85"])]
ETH_WHITE    = "Ethnic group: White"
ECON_ACT_COL = "Economic activity status: Economically active (excluding full-time students)"
UNEMP_COL    = "Economic activity status: Economically active (excluding full-time students): Unemployed"
RETIRED_COL  = "Economic activity status: Economically inactive: Retired"
LTSICK_COL   = "Economic activity status: Economically inactive: Long-term sick or disabled"

verification_results = {}
all_checks_passed = True

for lsoa, label in TEST_LSOAS.items():
    print(f"\n  {'='*66}")
    print(f"  {lsoa} — {label}")
    print(f"  {'='*66}")

    r_imd    = imd_df[imd_df["LSOA code (2021)"] == lsoa].iloc[0]
    r_nomis  = nomis_df[nomis_df["geography code"] == lsoa].iloc[0]
    r_census = census_df[census_df["geography code"] == lsoa].iloc[0]
    r_ruc    = ruc_df[ruc_df["LSOA21CD"] == lsoa].iloc[0]
    r_age    = age_df[age_df["geography code"] == lsoa].iloc[0]
    r_cars   = cars_df[cars_df["geography code"] == lsoa].iloc[0]
    r_eth    = eth_df[eth_df["geography code"] == lsoa].iloc[0]

    pop_total    = int(r_census["Residence type: Total; measures: Value"])
    econ_active  = int(r_nomis[ECON_ACT_COL])
    unemployed   = int(r_nomis[UNEMP_COL])
    unemp_rate   = unemployed / econ_active * 100 if econ_active > 0 else 0
    age_total    = int(r_age["Age: Total"])
    elderly      = int(sum(r_age[c] for c in AGE_65_COLS))
    households   = int(r_cars["Number of cars or vans: Total: All households"])
    no_car       = int(r_cars["Number of cars or vans: No cars or vans in household"])
    eth_total    = int(r_eth["Ethnic group: Total: All usual residents"])
    eth_white    = int(r_eth[ETH_WHITE])
    imd_score    = float(r_imd["Index of Multiple Deprivation (IMD) Score"])
    imd_rank     = int(r_imd["Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)"])
    imd_decile   = int(r_imd["Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"])
    income_score = float(r_imd["Income Score (rate)"])
    emp_score    = float(r_imd["Employment Score (rate)"])
    health_score = float(r_imd["Health Deprivation and Disability Score"])
    n_stops      = stop_counts.get(lsoa, 0)

    # ── Print full profile ─────────────────────────────────────────────────
    print(f"\n  POPULATION")
    print(f"    Census 2021:      {pop_total:,}")
    print(f"    Age dataset:      {age_total:,}   (diff={abs(pop_total-age_total)})")
    print(f"    Ethnicity total:  {eth_total:,}   (diff={abs(pop_total-eth_total)})")
    print(f"    IMD mid-2022 est: {int(r_imd['Total population: mid 2022']):,}   (different reference year)")

    print(f"\n  DEPRIVATION (IMD 2025 — England mean=21.7)")
    print(f"    IMD Score:        {imd_score:.3f}")
    print(f"    IMD Rank:         {imd_rank:,} / 33,755  (1=most deprived)")
    print(f"    IMD Decile:       {imd_decile}  (1=most deprived 10%)")
    print(f"    Income Score:     {income_score:.3f}  (proportion income-deprived, 0–1)")
    print(f"    Employment Score: {emp_score:.3f}  (proportion employment-deprived, 0–1)")
    print(f"    Health Score:     {health_score:.3f}  (z-score: negative=healthier than average)")
    print(f"    Living Env:       {float(r_imd['Living Environment Score']):.3f}")
    print(f"    Crime Score:      {float(r_imd['Crime Score']):.3f}")

    print(f"\n  EMPLOYMENT (Census 2021 TS066 — England mean=5.0%)")
    print(f"    Econ active:      {econ_active:,}  (excludes full-time students)")
    print(f"    Unemployed:       {unemployed:,}")
    print(f"    Unemployment:     {unemp_rate:.2f}%")
    print(f"    Retired:          {int(r_nomis[RETIRED_COL]):,}")
    print(f"    Long-term sick:   {int(r_nomis[LTSICK_COL]):,}")

    print(f"\n  DEMOGRAPHICS")
    print(f"    Elderly 65+:      {elderly:,} / {age_total:,} = {elderly/age_total*100:.1f}%")
    print(f"    No-car HH:        {no_car:,} / {households:,} = {no_car/households*100:.1f}%")
    print(f"    Non-white:        {eth_total-eth_white:,} / {eth_total:,} = {(eth_total-eth_white)/eth_total*100:.1f}%")
    print(f"    Urban/Rural:      {r_ruc['RUC21CD']} — {r_ruc['RUC21NM']}")

    print(f"\n  TRANSPORT")
    print(f"    Bus stops:        {n_stops}")
    if n_stops > 0:
        names = stop_names_d.get(lsoa, [])[:3]
        print(f"    Sample stops:     {names}")

    # ── Cross-validation checks ────────────────────────────────────────────
    checks = [
        ("Census pop ≈ Age total   (diff≤5)",   abs(pop_total - age_total) <= 5),
        ("Census pop ≈ Eth total   (diff≤10)",  abs(pop_total - eth_total) <= 10),
        ("Unemployed ≤ EconActive",             unemployed <= econ_active),
        ("NoCar ≤ Households",                  no_car <= households),
        ("Elderly ≤ AgePop",                    elderly <= age_total),
        ("IMD rank in [1, 33755]",              1 <= imd_rank <= 33755),
        ("IMD decile in [1, 10]",               1 <= imd_decile <= 10),
        ("Income score in [0, 1]",              0 <= income_score <= 1),
        ("Employment score in [0, 1]",          0 <= emp_score <= 1),
        ("Eth white ≤ total",                   eth_white <= eth_total),
    ]
    print(f"\n  CROSS-VALIDATION")
    lsoa_pass = True
    for name, result in checks:
        status = "✓" if result else "✗ FAIL"
        if not result:
            lsoa_pass = False
            all_checks_passed = False
        print(f"    {status}  {name}")

    verification_results[lsoa] = {
        "label":           label,
        "pop_total":       pop_total,
        "imd_score":       imd_score,
        "imd_rank":        imd_rank,
        "imd_decile":      imd_decile,
        "unemp_rate_pct":  round(unemp_rate, 2),
        "elderly_pct":     round(elderly/age_total*100, 1),
        "no_car_pct":      round(no_car/households*100, 1),
        "nonwhite_pct":    round((eth_total-eth_white)/eth_total*100, 1),
        "urban_rural":     r_ruc["RUC21CD"],
        "bus_stops":       n_stops,
        "all_checks_pass": lsoa_pass,
    }

print(f"\n  {'='*66}")
if all_checks_passed:
    print(f"  ✓ ALL CROSS-VALIDATION CHECKS PASSED across 4 LSOAs")
else:
    print(f"  ⚠ SOME CHECKS FAILED — review above")
print(f"  {'='*66}")

GT["single_lsoa_verification"] = verification_results
print(f"\n  ✓ Single-LSOA verification complete")



  10a. Single-LSOA Deep Verification — 4 Representative LSOAs



  E01000001 — City of London 001A — Wealthy urban core

  POPULATION
    Census 2021:      1,475
    Age dataset:      1,473   (diff=2)
    Ethnicity total:  1,474   (diff=1)
    IMD mid-2022 est: 1,795   (different reference year)

  DEPRIVATION (IMD 2025 — England mean=21.7)
    IMD Score:        8.742
    IMD Rank:         26,525 / 33,755  (1=most deprived)
    IMD Decile:       8  (1=most deprived 10%)
    Income Score:     0.013  (proportion income-deprived, 0–1)
    Employment Score: 0.014  (proportion employment-deprived, 0–1)
    Health Score:     -1.771  (z-score: negative=healthier than average)
    Living Env:       69.345
    Crime Score:      -2.220

  EMPLOYMENT (Census 2021 TS066 — England mean=5.0%)
    Econ active:      887  (excludes full-time students)
    Unemployed:       32
    Unemployment:     3.61%
    Retired:          305
    Long-term sick:   9

  DEMOGRAPHICS
    Elderly 65+:      370 / 1,473 = 25.1%
    No-car HH:        555 / 837 = 66.3%
    Non-white:  

In [21]:
section("10b. Full Dataset Column Profiling — Every Column Classified")

import zipfile
from pathlib import Path

DOCS_DIR = ROOT / "docs" / "data-dictionary"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

def classify_column(col_name, series, join_keys, use_cols, ignore_patterns):
    """Return USE / JOIN KEY / IGNORE with reason."""
    cn = col_name.lower()
    if col_name in join_keys:
        return "JOIN KEY"
    if col_name in use_cols:
        return "USE"
    for pat in ignore_patterns:
        if pat.lower() in cn:
            return f"IGNORE ({pat})"
    null_pct = series.isna().mean() * 100
    if null_pct == 100:
        return "IGNORE (100% null)"
    if null_pct > 95:
        return f"IGNORE ({null_pct:.0f}% null)"
    return "REVIEW"

def write_data_dict(name, df, join_keys, use_cols, ignore_patterns, unit_of_obs, traps, notes=""):
    """Write a markdown data dictionary for one dataset."""
    lines = [
        f"# Data Dictionary: {name}",
        f"",
        f"**Unit of observation:** {unit_of_obs}  ",
        f"**Rows:** {len(df):,}  ",
        f"**Columns:** {len(df.columns)}  ",
        f"",
        f"## Columns",
        f"",
        f"| Column | Type | Nulls% | Unique | Range/Values | Role |",
        f"|--------|------|--------|--------|--------------|------|",
    ]
    for col in df.columns:
        s = df[col]
        null_pct = s.isna().mean() * 100
        n_unique = s.nunique()
        dtype = str(s.dtype)
        if s.dtype in ("float64", "int64") and s.notna().sum() > 0:
            rng = f"{s.min():.2g} – {s.max():.2g}"
        else:
            top = s.dropna().value_counts().head(3).index.tolist()
            rng = str(top)[:40]
        role = classify_column(col, s, join_keys, use_cols, ignore_patterns)
        col_short = col[:55] if len(col) > 55 else col
        lines.append(f"| `{col_short}` | {dtype} | {null_pct:.1f}% | {n_unique:,} | {rng} | **{role}** |")

    if traps:
        lines += ["", "## Traps", ""]
        for trap in traps:
            lines.append(f"- {trap}")

    if notes:
        lines += ["", "## Notes", "", notes]

    out_path = DOCS_DIR / f"{name.lower().replace(' ', '-').replace('/', '-')}.md"
    out_path.write_text("\n".join(lines))
    return out_path

# ── NaPTAN ──────────────────────────────────────────────────────────────────
naptan_full = pd.read_csv(NAPTAN_DIR / "Stops.csv", low_memory=False)
p = write_data_dict(
    "NaPTAN Stops",
    naptan_full,
    join_keys=["ATCOCode", "NaptanCode"],
    use_cols=["ATCOCode","NaptanCode","CommonName","Latitude","Longitude",
              "StopType","BusStopType","Status","Easting","Northing",
              "Indicator","Bearing","LocalityName","AdministrativeAreaCode",
              "TimingStatus","CreationDateTime","ModificationDateTime"],
    ignore_patterns=["Lang","CleardownCode","DefaultWaitTime","Notes",
                     "Crossing","GrandParent","ShortCommon"],
    unit_of_obs="One row = one physical bus stop location",
    traps=[
        "StopType must be filtered to BCT/BCS/BCE for bus stops only (18 stop types exist)",
        "Status must be filtered to 'active' (inactive=45,840, pending=1)",
        "ATCO prefix 0xx-4xx = England only; 5xx=Wales, 6xx=Scotland",
        "52,449 rows have null Longitude/Latitude — use Easting/Northing as fallback",
        "NaptanCode → BODS stop_code join (NOT ATCOCode → stop_id)",
        "Multiple stops share CommonName (e.g. 'Temple Meads Stn' = 7 different stops)",
        "BusStopType null for non-bus stop types — filter StopType first",
    ],
)
print(f"  Written: {p}")

# ── IMD 2025 ─────────────────────────────────────────────────────────────────
p = write_data_dict(
    "IMD 2025",
    imd_df,
    join_keys=["LSOA code (2021)"],
    use_cols=[
        "LSOA code (2021)", "LSOA name (2021)",
        "Index of Multiple Deprivation (IMD) Score",
        "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
        "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
        "Income Score (rate)", "Employment Score (rate)",
        "Education, Skills and Training Score",
        "Health Deprivation and Disability Score",
        "Crime Score", "Barriers to Housing and Services Score",
        "Living Environment Score",
        "Income Deprivation Affecting Children Index (IDACI) Score (rate)",
        "Income Deprivation Affecting Older People (IDAOPI) Score (rate)",
        "Total population: mid 2022",
        "Local Authority District code (2024)", "Local Authority District name (2024)",
    ],
    ignore_patterns=["Rank (where", "Decile (where", "Sub-domain Rank", "Sub-domain Decile",
                     "Dependent Children", "Older population", "Working age population"],
    unit_of_obs="One row = one LSOA (2021 boundaries, England only)",
    traps=[
        "Decile 1 = MOST deprived 10% of LSOAs (not least deprived)",
        "Rank 1 = nationally most deprived LSOA; rank 33,755 = least deprived",
        "Domain scores (Income, Employment etc) are NOT additive to IMD Score",
        "Income Score is a rate (0-1), not an absolute value",
        "Health Score is a z-score — negative values = healthier than average",
        "Living Environment Score range is wide (0-100+) unlike other domain scores",
        "LAD codes use 2024 boundaries; LSOA codes use 2021 boundaries",
        "Population denominator is mid-2022 estimate, not Census 2021 count",
    ],
)
print(f"  Written: {p}")

# ── NOMIS TS066 ───────────────────────────────────────────────────────────────
p = write_data_dict(
    "NOMIS TS066 Unemployment",
    nomis_df[nomis_df["geography code"].str.startswith("E", na=False)],
    join_keys=["geography code"],
    use_cols=[
        "geography code", "geography",
        "Economic activity status: Total: All usual residents aged 16 years and over",
        "Economic activity status: Economically active (excluding full-time students)",
        "Economic activity status: Economically active (excluding full-time students): Unemployed",
        "Economic activity status: Economically inactive",
        "Economic activity status: Economically inactive: Retired",
        "Economic activity status: Economically inactive: Long-term sick or disabled",
        "Economic activity status: Economically inactive: Looking after home or family",
    ],
    ignore_patterns=["full-time student:", "Self-employed", "Employee:", "Part-time", "Full-time"],
    unit_of_obs="One row = one LSOA (2021 boundaries, England+Wales — filter to E*)",
    traps=[
        "Raw file has 35,672 rows (England 33,755 + Wales 1,917) — ALWAYS filter geography code LIKE 'E%'",
        "Unemployment rate = unemployed / economically_active (NOT unemployed / total_pop)",
        "Full-time students who work are classified separately — excluded from main econ-active denominator",
        "Long-term sick count is a useful proxy for health deprivation (cross-validate with IMD health score)",
        "Column names are extremely long — truncate for pipeline use",
    ],
)
print(f"  Written: {p}")

# ── Census TS001 Population ───────────────────────────────────────────────────
p = write_data_dict(
    "Census TS001 Population",
    census_df[census_df["geography code"].str.startswith("E", na=False)],
    join_keys=["geography code"],
    use_cols=["geography code", "geography",
              "Residence type: Total; measures: Value"],
    ignore_patterns=["household", "communal"],
    unit_of_obs="One row = one LSOA (England+Wales — filter to E*)",
    traps=[
        "Raw file has 35,672 rows — filter geography code to E* for England-only",
        "Total population includes communal establishments (prisons, care homes) — use Total not household",
        "This is ALL ages; NOMIS denominators are age 16+ — never mix for rate calculations",
        "Population figures are Census night 21 March 2021 — not mid-year estimates",
    ],
)
print(f"  Written: {p}")

# ── RUC 2021 ──────────────────────────────────────────────────────────────────
p = write_data_dict(
    "Rural Urban Classification 2021",
    ruc_df[ruc_df["LSOA21CD"].str.startswith("E", na=False)],
    join_keys=["LSOA21CD"],
    use_cols=["LSOA21CD", "RUC21CD", "RUC21NM", "Urban_rural_flag"],
    ignore_patterns=["NMW", "LSOA21NM"],
    unit_of_obs="One row = one LSOA (England+Wales — filter to E*)",
    traps=[
        "Urban_rural_flag is binary (Urban/Rural) — use RUC21CD for 6-category analysis",
        "6 categories: UN1 urban-core, UF1 urban-fringe, RSN1 rural-small-town, RLN1 rural-less-sparse, RSF1 rural-sparse, RLF1 rural-very-remote",
        "Classification is distance-based (proximity to nearest major town), NOT density-based",
        "RSN1 (Small rural towns near cities) is classified Rural despite urban proximity",
        "LSOA21NMW (Welsh names) is 94.6% null — ignore entirely",
    ],
)
print(f"  Written: {p}")

# ── Census TS007a Age ─────────────────────────────────────────────────────────
with zipfile.ZipFile(CENSUS_DIR / "census2021-ts007a.zip") as zf:
    age_df_full = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
age_eng = age_df_full[age_df_full["geography code"].str.startswith("E", na=False)]

age_use_cols = ["geography code", "geography", "Age: Total"] + AGE_65_COLS + [
    c for c in age_df_full.columns if any(x in c for x in ["0 to","4 years","9 years","14 years"])
]
p = write_data_dict(
    "Census TS007a Age",
    age_eng,
    join_keys=["geography code"],
    use_cols=list(set(age_use_cols)),
    ignore_patterns=["date"],
    unit_of_obs="One row = one LSOA (England+Wales — filter to E*)",
    traps=[
        "Raw file has 35,672 rows (E+W) — filter to E*",
        "'Age: Aged 85 years and over' is unbounded — cannot distinguish 85 from 100",
        "Elderly 65+ = sum of columns containing 65, 70, 75, 80, 85 in column name",
        "Age bands are 5-year intervals — cannot subdivide further",
        "Total should equal Census TS001 total within ±5 (rounding differences)",
    ],
)
print(f"  Written: {p}")

# ── Census TS021 Ethnicity ────────────────────────────────────────────────────
with zipfile.ZipFile(CENSUS_DIR / "census2021-ts021.zip") as zf:
    eth_df_full = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
eth_eng = eth_df_full[eth_df_full["geography code"].str.startswith("E", na=False)]
p = write_data_dict(
    "Census TS021 Ethnicity",
    eth_eng,
    join_keys=["geography code"],
    use_cols=["geography code", "geography", "Ethnic group: Total: All usual residents",
              "Ethnic group: White",
              "Ethnic group: Asian, Asian British or Asian Welsh",
              "Ethnic group: Black, Black British, Black Welsh, Caribbean or African",
              "Ethnic group: Mixed or Multiple ethnic groups",
              "Ethnic group: Other ethnic group"],
    ignore_patterns=["date", ": Bangladeshi", ": Chinese", ": Indian", ": Pakistani",
                     ": African", ": Caribbean", "English, Welsh", "Irish", "Gypsy"],
    unit_of_obs="One row = one LSOA (England+Wales — filter to E*)",
    traps=[
        "Raw file has 35,672 rows (E+W) — filter to E*",
        "Hierarchy: group totals should >= sum of subcategories",
        "2021 added Roma explicitly — NOT comparable to 2011 Census ethnicity data",
        "'White: Other White' is a residual catch-all (Polish, Italian, etc.) — not homogeneous",
        "Census 2021 allowed multiple-tick for mixed groups — some double-counting possible at sub-category level",
        "Non-white % = (Total - White) / Total",
    ],
)
print(f"  Written: {p}")

# ── Census TS045 Car Ownership ────────────────────────────────────────────────
with zipfile.ZipFile(CENSUS_DIR / "census2021-ts045.zip") as zf:
    cars_df_full = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
cars_eng = cars_df_full[cars_df_full["geography code"].str.startswith("E", na=False)]
p = write_data_dict(
    "Census TS045 Car Ownership",
    cars_eng,
    join_keys=["geography code"],
    use_cols=["geography code", "geography",
              "Number of cars or vans: Total: All households",
              "Number of cars or vans: No cars or vans in household",
              "Number of cars or vans: 1 car or van in household"],
    ignore_patterns=["date", "2 cars", "3 or more"],
    unit_of_obs="One row = one LSOA (England+Wales — filter to E*)",
    traps=[
        "Raw file has 35,672 rows (E+W) — filter to E*",
        "HOUSEHOLD-level, NOT person-level — do not divide by population",
        "High no-car % = wealthy city centre (London) OR poor deprived area — NOT a simple poverty signal",
        "'3 or more' is unbounded — no distinction between 3 and 10 cars",
        "Includes vans — some households are commercial, skews rural LSOAs",
        "No-car rate = no_cars / total_households (not per person)",
    ],
)
print(f"  Written: {p}")

print(f"\n  ✓ All data dictionaries written to {DOCS_DIR}")
print(f"  Files: {[f.name for f in DOCS_DIR.glob('*.md')]}")
GT["data_dictionary_generated"] = True
GT["data_dictionary_path"] = str(DOCS_DIR.relative_to(ROOT))



  10b. Full Dataset Column Profiling — Every Column Classified


  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/naptan-stops.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/imd-2025.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/nomis-ts066-unemployment.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/census-ts001-population.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/rural-urban-classification-2021.md


  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/census-ts007a-age.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/census-ts021-ethnicity.md
  Written: /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary/census-ts045-car-ownership.md

  ✓ All data dictionaries written to /Users/souravamseekarmarti/Projects/aequitas/docs/data-dictionary
  Files: ['nomis-ts066-unemployment.md', 'imd-2025.md', 'census-ts007a-age.md', 'rural-urban-classification-2021.md', 'census-ts021-ethnicity.md', 'naptan-stops.md', 'census-ts001-population.md', 'census-ts045-car-ownership.md']


In [22]:
section("10c. Join Integrity — Full Cross-Dataset Referential Integrity")

print("  Validating all dataset joins on full 33,755 England LSOAs...\n")

# Canonical LSOA set from IMD (England-only by definition)
canonical = set(imd_df["LSOA code (2021)"].dropna())
print(f"  Canonical set (IMD 2025):    {len(canonical):,} LSOAs")

datasets = {
    "NOMIS TS066":    (nomis_df[nomis_df["geography code"].str.startswith("E", na=False)], "geography code"),
    "Census TS001":   (census_df[census_df["geography code"].str.startswith("E", na=False)], "geography code"),
    "RUC 2021":       (ruc_df[ruc_df["LSOA21CD"].str.startswith("E", na=False)], "LSOA21CD"),
    "Age TS007a":     (age_df[age_df["geography code"].str.startswith("E", na=False)], "geography code"),
    "Cars TS045":     (cars_df[cars_df["geography code"].str.startswith("E", na=False)], "geography code"),
    "Ethnicity TS021":(eth_df[eth_df["geography code"].str.startswith("E", na=False)], "geography code"),
}

join_results = {}
all_clean = True
print(f"  {'Dataset':<22} {'Rows':>7} {'In IMD':>8} {'Not in IMD':>10} {'Missing from DS':>15} {'Status'}")
print(f"  {'-'*80}")

for name, (df, key_col) in datasets.items():
    ds_keys   = set(df[key_col].dropna())
    in_imd    = ds_keys & canonical
    not_in_imd = ds_keys - canonical
    missing   = canonical - ds_keys
    ok        = len(not_in_imd) == 0 and len(missing) == 0
    if not ok:
        all_clean = False
    status = "✓ Clean" if ok else "⚠ Issues"
    print(f"  {name:<22} {len(ds_keys):>7,} {len(in_imd):>8,} {len(not_in_imd):>10,} {len(missing):>15,} {status}")
    join_results[name] = {
        "rows": len(ds_keys),
        "in_imd": len(in_imd),
        "not_in_imd": len(not_in_imd),
        "missing_from_ds": len(missing),
        "clean": ok,
    }

print(f"  {'-'*80}")
print(f"\n  {'✓ All joins clean' if all_clean else '⚠ Some joins have issues — review above'}")

# Duplicate key check
print(f"\n  Duplicate key check:")
for name, (df, key_col) in datasets.items():
    dupes = df[key_col].duplicated().sum()
    print(f"    {name:<22} duplicates on key: {dupes} {'✓' if dupes==0 else '✗ PROBLEM'}")

GT["join_integrity"] = join_results
GT["join_integrity_all_clean"] = all_clean
print(f"\n  ✓ Join integrity validation complete")



  10c. Join Integrity — Full Cross-Dataset Referential Integrity
  Validating all dataset joins on full 33,755 England LSOAs...

  Canonical set (IMD 2025):    33,755 LSOAs


  Dataset                   Rows   In IMD Not in IMD Missing from DS Status
  --------------------------------------------------------------------------------
  NOMIS TS066             33,755   33,755          0               0 ✓ Clean
  Census TS001            33,755   33,755          0               0 ✓ Clean
  RUC 2021                33,755   33,755          0               0 ✓ Clean
  Age TS007a              33,755   33,755          0               0 ✓ Clean
  Cars TS045              33,755   33,755          0               0 ✓ Clean
  Ethnicity TS021         33,755   33,755          0               0 ✓ Clean
  --------------------------------------------------------------------------------

  ✓ All joins clean

  Duplicate key check:
    NOMIS TS066            duplicates on key: 0 ✓
    Census TS001           duplicates on key: 0 ✓
    RUC 2021               duplicates on key: 0 ✓
    Age TS007a             duplicates on key: 0 ✓
    Cars TS045             duplicates on key: 0 ✓
 

# 11. Exhaustive Data Quality Investigation

**Coverage:** Every data science technique applicable to this dataset collection.

Categories covered:
1. Completeness — nulls, empty strings, whitespace-only values
2. Validity — range constraints, categorical domain, format
3. Uniqueness — exact duplicates, near-duplicates, key violations
4. Mathematical consistency — column relationships that must hold by definition
5. Cross-dataset consistency — same LSOA across all datasets must agree on population
6. Statistical profiling — skewness, kurtosis, outlier detection (IQR + Z-score + Isolation Forest)
7. Distribution sanity — do distributions match domain knowledge?
8. Correlation analysis — are inter-variable relationships physically sensible?
9. Benford's Law — leading digit analysis for count columns
10. Spatial quality — geometry validity, coordinate precision, topology
11. Cardinality analysis — suspiciously low/high unique counts
12. Encoding/structural anomalies — header rows repeated, mixed types, encoding errors
13. Business rule validation — domain-specific rules that must hold
14. Accuracy cross-check — verify against ONS published aggregate statistics

**Output:** Structured audit log — every failure with severity (CRITICAL/HIGH/MEDIUM/LOW), dataset, column, affected row count, and description.


In [23]:
section("11. Setup — Audit Infrastructure")

import zipfile
import warnings
import scipy.stats as stats
from sklearn.ensemble import IsolationForest
warnings.filterwarnings("ignore")

# ── Load all datasets (England-only) ──────────────────────────────────────
imd_df    = pd.read_csv(IMD_DIR / "imd2025_all_ranks_scores_deciles.csv")
nomis_df  = pd.read_csv(NOMIS_DIR / "nomis_unemployment_lsoa.csv")
nomis_df  = nomis_df[nomis_df["geography code"].str.startswith("E", na=False)].copy()
census_df = pd.read_csv(CENSUS_DIR / "census2021_ts001_lsoa_population.csv")
census_df = census_df[census_df["geography code"].str.startswith("E", na=False)].copy()
ruc_df    = pd.read_csv(CENSUS_DIR / "ruc2021_lsoa_ew.csv")
ruc_df    = ruc_df[ruc_df["LSOA21CD"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(CENSUS_DIR / "census2021-ts007a.zip") as zf:
    age_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
age_df = age_df[age_df["geography code"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(CENSUS_DIR / "census2021-ts045.zip") as zf:
    cars_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
cars_df = cars_df[cars_df["geography code"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(CENSUS_DIR / "census2021-ts021.zip") as zf:
    eth_df = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
eth_df = eth_df[eth_df["geography code"].str.startswith("E", na=False)].copy()

naptan_df = pd.read_csv(NAPTAN_DIR / "Stops.csv", low_memory=False)
naptan_eng = naptan_df[
    naptan_df["StopType"].isin({"BCT","BCS","BCE"}) &
    (naptan_df["Status"] == "active") &
    naptan_df["ATCOCode"].str.match(r"^[01234]", na=False) &
    naptan_df["Latitude"].notna()
].copy()

# ── Audit log ─────────────────────────────────────────────────────────────
AUDIT_LOG = []

def log(check_id, category, dataset, column, severity, status, observed, expected, affected_rows, affected_pct, notes):
    AUDIT_LOG.append({
        "check_id": check_id, "category": category, "dataset": dataset,
        "column": column, "severity": severity, "status": status,
        "observed": str(observed)[:80], "expected": str(expected)[:80],
        "affected_rows": int(affected_rows), "affected_pct": round(float(affected_pct), 4),
        "notes": notes,
    })
    icon = {"PASS": "✓", "FAIL": "✗", "WARN": "⚠"}.get(status, "·")
    sev_icon = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢", "INFO": "⚪"}.get(severity, "")
    if status != "PASS":
        print(f"  {icon} {sev_icon} [{check_id}] {dataset}/{column}: {notes} (rows={affected_rows}, {affected_pct:.1f}%)")

CANONICAL_LSOAS = set(imd_df["LSOA code (2021)"])
N = len(CANONICAL_LSOAS)
print(f"  Canonical LSOA set: {N:,}")
print(f"  All datasets loaded. Audit log initialised.")
print(f"  Only FAILs and WARNs printed inline — see audit_log DataFrame for full results.")



  11. Setup — Audit Infrastructure


  Canonical LSOA set: 33,755
  All datasets loaded. Audit log initialised.
  Only FAILs and WARNs printed inline — see audit_log DataFrame for full results.


In [24]:
section("11a. Completeness — Nulls, Empty Strings, Whitespace")

DATASETS = {
    "IMD2025":   (imd_df,    "LSOA code (2021)"),
    "NOMIS":     (nomis_df,  "geography code"),
    "Census":    (census_df, "geography code"),
    "RUC":       (ruc_df,    "LSOA21CD"),
    "Age":       (age_df,    "geography code"),
    "Cars":      (cars_df,   "geography code"),
    "Ethnicity": (eth_df,    "geography code"),
}

for ds_name, (df, key_col) in DATASETS.items():
    # 1. Null check on join key
    null_keys = df[key_col].isna().sum()
    log(f"C1.{ds_name}.key_null", "Completeness", ds_name, key_col,
        "CRITICAL" if null_keys > 0 else "INFO",
        "FAIL" if null_keys > 0 else "PASS",
        null_keys, 0, null_keys, null_keys/len(df)*100,
        f"Null join keys")

    # 2. Empty string on join key
    empty_keys = (df[key_col].fillna("") == "").sum()
    log(f"C2.{ds_name}.key_empty", "Completeness", ds_name, key_col,
        "CRITICAL" if empty_keys > 0 else "INFO",
        "FAIL" if empty_keys > 0 else "PASS",
        empty_keys, 0, empty_keys, empty_keys/len(df)*100,
        "Empty string join keys")

    # 3. Whitespace-only on join key
    ws_keys = df[key_col].astype(str).str.strip().ne(df[key_col].astype(str)).sum()
    log(f"C3.{ds_name}.key_whitespace", "Completeness", ds_name, key_col,
        "HIGH" if ws_keys > 0 else "INFO",
        "FAIL" if ws_keys > 0 else "PASS",
        ws_keys, 0, ws_keys, ws_keys/len(df)*100,
        "Whitespace around join keys")

    # 4. Null counts on ALL numeric columns
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        n_null = df[col].isna().sum()
        if n_null > 0:
            log(f"C4.{ds_name}.{col[:30]}", "Completeness", ds_name, col,
                "HIGH", "FAIL",
                n_null, 0, n_null, n_null/len(df)*100,
                f"Nulls in numeric column")

    # 5. All-zero columns (may indicate missing data coded as 0)
    for col in num_cols:
        if df[col].notna().sum() > 0 and (df[col].fillna(0) == 0).all():
            log(f"C5.{ds_name}.{col[:30]}", "Completeness", ds_name, col,
                "MEDIUM", "WARN",
                "all zeros", "non-zero expected", len(df), 100,
                "Column is entirely zero — may be missing data coded as 0")

# NaPTAN completeness
naptan_key_nulls = naptan_df["ATCOCode"].isna().sum()
log("C1.NaPTAN.ATCOCode", "Completeness", "NaPTAN", "ATCOCode",
    "CRITICAL" if naptan_key_nulls > 0 else "INFO",
    "FAIL" if naptan_key_nulls > 0 else "PASS",
    naptan_key_nulls, 0, naptan_key_nulls, naptan_key_nulls/len(naptan_df)*100,
    "Null ATCOCode in NaPTAN")

coord_nulls = naptan_df["Latitude"].isna().sum()
log("C6.NaPTAN.coords", "Completeness", "NaPTAN", "Latitude",
    "HIGH" if coord_nulls > 0 else "INFO",
    "WARN" if coord_nulls > 0 else "PASS",
    coord_nulls, 0, coord_nulls, coord_nulls/len(naptan_df)*100,
    f"Null coordinates in raw NaPTAN ({coord_nulls:,} rows — filtered out in England set)")

passes = sum(1 for r in AUDIT_LOG if r["status"]=="PASS" and r["category"]=="Completeness")
fails  = sum(1 for r in AUDIT_LOG if r["status"]!="PASS" and r["category"]=="Completeness")
print(f"\n  Completeness: {passes} PASS, {fails} FAIL/WARN")



  11a. Completeness — Nulls, Empty Strings, Whitespace
  ⚠ 🟠 [C6.NaPTAN.coords] NaPTAN/Latitude: Null coordinates in raw NaPTAN (52,449 rows — filtered out in England set) (rows=52449, 12.1%)

  Completeness: 22 PASS, 1 FAIL/WARN


In [25]:
section("11b. Validity — Range, Domain, Format Constraints")

# IMD Score range: physically 0–100+ (min observed ~0.2, max ~94)
imd_score = imd_df["Index of Multiple Deprivation (IMD) Score"]
invalid_imd = (imd_score < 0).sum()
log("V1.IMD.score_range", "Validity", "IMD2025", "IMD Score",
    "CRITICAL" if invalid_imd > 0 else "INFO", "FAIL" if invalid_imd > 0 else "PASS",
    f"min={imd_score.min():.3f}", ">=0", invalid_imd, invalid_imd/N*100,
    "IMD scores below 0 are impossible")

# IMD Rank: must be 1–33755, all unique
imd_rank = imd_df["Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)"]
rank_out  = ((imd_rank < 1) | (imd_rank > N)).sum()
rank_dupe = imd_rank.duplicated().sum()
log("V2.IMD.rank_range", "Validity", "IMD2025", "IMD Rank", "CRITICAL" if rank_out>0 else "INFO",
    "FAIL" if rank_out>0 else "PASS", f"range [{imd_rank.min()},{imd_rank.max()}]", f"[1,{N}]",
    rank_out, rank_out/N*100, "IMD rank outside valid range")
log("V3.IMD.rank_unique", "Validity", "IMD2025", "IMD Rank", "CRITICAL" if rank_dupe>0 else "INFO",
    "FAIL" if rank_dupe>0 else "PASS", f"{rank_dupe} dupes", "0 dupes",
    rank_dupe, rank_dupe/N*100, "Duplicate IMD ranks (must be bijection 1..33755)")

# IMD Decile: must be 1–10 only
imd_decile = imd_df["Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"]
bad_decile = (~imd_decile.isin(range(1,11))).sum()
log("V4.IMD.decile_domain", "Validity", "IMD2025", "IMD Decile", "CRITICAL" if bad_decile>0 else "INFO",
    "FAIL" if bad_decile>0 else "PASS", imd_decile.unique().tolist(), list(range(1,11)),
    bad_decile, bad_decile/N*100, "IMD decile outside [1,10]")

# Income/Employment Score: must be 0–1 (rates)
for col, label in [
    ("Income Score (rate)", "Income"),
    ("Employment Score (rate)", "Employment"),
]:
    s = imd_df[col]
    bad = ((s < 0) | (s > 1)).sum()
    log(f"V5.IMD.{label}_rate", "Validity", "IMD2025", col, "CRITICAL" if bad>0 else "INFO",
        "FAIL" if bad>0 else "PASS", f"[{s.min():.4f},{s.max():.4f}]", "[0,1]",
        bad, bad/N*100, f"{label} score outside valid rate range [0,1]")

# NOMIS: unemployment rate
econ = nomis_df["Economic activity status: Economically active (excluding full-time students)"]
unemp = nomis_df["Economic activity status: Economically active (excluding full-time students): Unemployed"]
bad_unemp = (unemp > econ).sum()
neg_econ  = (econ < 0).sum()
log("V6.NOMIS.unemp_le_active", "Validity", "NOMIS", "Unemployed",
    "CRITICAL" if bad_unemp>0 else "INFO", "FAIL" if bad_unemp>0 else "PASS",
    f"{bad_unemp} violations", "unemployed <= econ_active always",
    bad_unemp, bad_unemp/N*100, "Unemployed > economically active — impossible")
log("V7.NOMIS.neg_econ_active", "Validity", "NOMIS", "Econ Active",
    "CRITICAL" if neg_econ>0 else "INFO", "FAIL" if neg_econ>0 else "PASS",
    f"{neg_econ} negatives", ">=0", neg_econ, neg_econ/N*100,
    "Negative economically active count")

# NOMIS: all count columns must be >= 0
count_cols = nomis_df.select_dtypes(include="number").columns
for col in count_cols:
    neg = (nomis_df[col] < 0).sum()
    if neg > 0:
        log(f"V8.NOMIS.{col[:25]}_neg", "Validity", "NOMIS", col,
            "HIGH", "FAIL", f"{neg} negatives", ">=0",
            neg, neg/N*100, "Negative count in census data")

# Census: total population must be positive
pop_col = "Residence type: Total; measures: Value"
bad_pop = (census_df[pop_col] <= 0).sum()
log("V9.Census.pop_positive", "Validity", "Census", pop_col,
    "CRITICAL" if bad_pop>0 else "INFO", "FAIL" if bad_pop>0 else "PASS",
    f"{bad_pop} non-positive", ">0", bad_pop, bad_pop/N*100,
    "Zero or negative population — LSOAs must have population > 0")

# Car ownership: no-car <= total households
hh_total = cars_df["Number of cars or vans: Total: All households"]
no_car   = cars_df["Number of cars or vans: No cars or vans in household"]
bad_car  = (no_car > hh_total).sum()
neg_hh   = (hh_total <= 0).sum()
log("V10.Cars.nocar_le_total", "Validity", "Cars", "No cars",
    "CRITICAL" if bad_car>0 else "INFO", "FAIL" if bad_car>0 else "PASS",
    f"{bad_car} violations", "no_car <= total_hh", bad_car, bad_car/N*100,
    "No-car households > total households — impossible")
log("V11.Cars.hh_positive", "Validity", "Cars", "Total HH",
    "HIGH" if neg_hh>0 else "INFO", "FAIL" if neg_hh>0 else "PASS",
    f"{neg_hh} non-positive", ">0", neg_hh, neg_hh/N*100,
    "Zero or negative household count")

# Age: all bands sum to total
age_band_cols = [c for c in age_df.columns if c.startswith("Age:") and c != "Age: Total"]
age_sum = age_df[age_band_cols].sum(axis=1)
age_total_col = age_df["Age: Total"]
age_mismatch = (abs(age_sum - age_total_col) > 2).sum()
log("V12.Age.bands_sum", "Validity", "Age", "Age bands",
    "CRITICAL" if age_mismatch>0 else "INFO", "FAIL" if age_mismatch>0 else "PASS",
    f"{age_mismatch} mismatches", "sum(bands) == Total ±2",
    age_mismatch, age_mismatch/N*100, "Age band sum != Age Total")

# Ethnicity: subgroups sum to total
eth_total_col = eth_df["Ethnic group: Total: All usual residents"]
eth_group_cols = [c for c in eth_df.columns
                  if c.startswith("Ethnic group:") and
                  c != "Ethnic group: Total: All usual residents" and
                  ":" not in c.replace("Ethnic group: ","")]
eth_sum  = eth_df[eth_group_cols].sum(axis=1)
eth_diff = (abs(eth_sum - eth_total_col) > 5).sum()
log("V13.Eth.groups_sum", "Validity", "Ethnicity", "Ethnic groups",
    "HIGH" if eth_diff>0 else "INFO", "WARN" if eth_diff>0 else "PASS",
    f"{eth_diff} mismatches", "sum(top-level groups) ≈ Total ±5",
    eth_diff, eth_diff/N*100, "Ethnic group sum deviates from total by >5")

# NaPTAN: coordinate bounds for England
eng_lat_bad = ((naptan_eng["Latitude"] < 49.9) | (naptan_eng["Latitude"] > 55.9)).sum()
eng_lon_bad = ((naptan_eng["Longitude"] < -6.5) | (naptan_eng["Longitude"] > 2.1)).sum()
log("V14.NaPTAN.lat_bounds", "Validity", "NaPTAN", "Latitude",
    "HIGH" if eng_lat_bad>0 else "INFO", "FAIL" if eng_lat_bad>0 else "PASS",
    f"{eng_lat_bad} outside [49.9,55.9]", "49.9–55.9", eng_lat_bad,
    eng_lat_bad/len(naptan_eng)*100, "Stop latitude outside England bounds")
log("V15.NaPTAN.lon_bounds", "Validity", "NaPTAN", "Longitude",
    "HIGH" if eng_lon_bad>0 else "INFO", "FAIL" if eng_lon_bad>0 else "PASS",
    f"{eng_lon_bad} outside [-6.5,2.1]", "-6.5–2.1", eng_lon_bad,
    eng_lon_bad/len(naptan_eng)*100, "Stop longitude outside England bounds")

# RUC: valid codes only
valid_ruc = {"UN1","UF1","RSN1","RLN1","RSF1","RLF1"}
bad_ruc = (~ruc_df["RUC21CD"].isin(valid_ruc)).sum()
log("V16.RUC.code_domain", "Validity", "RUC", "RUC21CD",
    "CRITICAL" if bad_ruc>0 else "INFO", "FAIL" if bad_ruc>0 else "PASS",
    ruc_df["RUC21CD"].unique().tolist(), list(valid_ruc),
    bad_ruc, bad_ruc/N*100, "RUC code outside valid 6-category domain")

# LSOA code format: must match E[0-9]{8}
import re
lsoa_fmt = re.compile(r"^E[0-9]{8}$")
bad_fmt_imd = (~imd_df["LSOA code (2021)"].str.match(r"^E[0-9]{8}$", na=False)).sum()
log("V17.IMD.lsoa_format", "Validity", "IMD2025", "LSOA code",
    "CRITICAL" if bad_fmt_imd>0 else "INFO", "FAIL" if bad_fmt_imd>0 else "PASS",
    f"{bad_fmt_imd} bad format", "E[0-9]{8}", bad_fmt_imd, bad_fmt_imd/N*100,
    "LSOA code format mismatch (should be E followed by 8 digits)")

fails_v = sum(1 for r in AUDIT_LOG if r["status"]!="PASS" and r["category"]=="Validity")
passes_v = sum(1 for r in AUDIT_LOG if r["status"]=="PASS" and r["category"]=="Validity")
print(f"\n  Validity: {passes_v} PASS, {fails_v} FAIL/WARN")



  11b. Validity — Range, Domain, Format Constraints

  Validity: 17 PASS, 0 FAIL/WARN


In [26]:
section("11c. Uniqueness — Exact Duplicates, Near-Duplicates, Key Violations")

for ds_name, (df, key_col) in DATASETS.items():
    # Exact duplicate rows
    exact_dupe_rows = df.duplicated().sum()
    log(f"U1.{ds_name}.exact_dupe_rows", "Uniqueness", ds_name, "ALL",
        "HIGH" if exact_dupe_rows>0 else "INFO",
        "FAIL" if exact_dupe_rows>0 else "PASS",
        exact_dupe_rows, 0, exact_dupe_rows, exact_dupe_rows/len(df)*100,
        "Exact duplicate rows")

    # Duplicate join keys
    dupe_keys = df[key_col].duplicated().sum()
    log(f"U2.{ds_name}.dupe_key", "Uniqueness", ds_name, key_col,
        "CRITICAL" if dupe_keys>0 else "INFO",
        "FAIL" if dupe_keys>0 else "PASS",
        dupe_keys, 0, dupe_keys, dupe_keys/len(df)*100,
        "Duplicate join keys — breaks 1:1 LSOA joins")

# NaPTAN: duplicate ATCOCodes across England-filtered set
naptan_dupe = naptan_eng["ATCOCode"].duplicated().sum()
log("U3.NaPTAN.dupe_atco", "Uniqueness", "NaPTAN", "ATCOCode",
    "CRITICAL" if naptan_dupe>0 else "INFO",
    "FAIL" if naptan_dupe>0 else "PASS",
    naptan_dupe, 0, naptan_dupe, naptan_dupe/len(naptan_eng)*100,
    "Duplicate ATCOCodes in England active bus stops")

# NaPTAN: near-duplicate stops (same lat/lon within 5m = 0.00005 degrees)
stops_sorted = naptan_eng.sort_values(["Latitude","Longitude"])
lat_diff = stops_sorted["Latitude"].diff().abs()
lon_diff = stops_sorted["Longitude"].diff().abs()
near_dupes = ((lat_diff < 0.00005) & (lon_diff < 0.00005)).sum()
log("U4.NaPTAN.near_dupe_coords", "Uniqueness", "NaPTAN", "Lat/Lon",
    "MEDIUM" if near_dupes>0 else "INFO",
    "WARN" if near_dupes>0 else "PASS",
    near_dupes, 0, near_dupes, near_dupes/len(naptan_eng)*100,
    f"Stops within 5m of each other ({near_dupes:,} — may be legitimate opposite-side-of-road stops)")

# IMD: duplicate LSOA codes
imd_dupe = imd_df["LSOA code (2021)"].duplicated().sum()
log("U5.IMD.dupe_lsoa", "Uniqueness", "IMD2025", "LSOA code",
    "CRITICAL" if imd_dupe>0 else "INFO",
    "FAIL" if imd_dupe>0 else "PASS",
    imd_dupe, 0, imd_dupe, imd_dupe/N*100,
    "Duplicate LSOA codes in IMD dataset")

fails_u = sum(1 for r in AUDIT_LOG if r["status"]!="PASS" and r["category"]=="Uniqueness")
passes_u = sum(1 for r in AUDIT_LOG if r["status"]=="PASS" and r["category"]=="Uniqueness")
print(f"\n  Uniqueness: {passes_u} PASS, {fails_u} FAIL/WARN")
print(f"  NaPTAN near-duplicate stops (within 5m): {near_dupes:,} — these are typically pole A/pole B on same junction")



  11c. Uniqueness — Exact Duplicates, Near-Duplicates, Key Violations


  ⚠ 🟡 [U4.NaPTAN.near_dupe_coords] NaPTAN/Lat/Lon: Stops within 5m of each other (604 — may be legitimate opposite-side-of-road stops) (rows=604, 0.2%)

  Uniqueness: 16 PASS, 1 FAIL/WARN
  NaPTAN near-duplicate stops (within 5m): 604 — these are typically pole A/pole B on same junction


In [27]:
section("11d. Statistical Profiling — Skewness, Kurtosis, IQR Outliers, Z-Score, Isolation Forest")

from scipy import stats as scipy_stats
from sklearn.ensemble import IsolationForest

KEY_METRICS = {
    "IMD Score":        imd_df["Index of Multiple Deprivation (IMD) Score"],
    "Income Score":     imd_df["Income Score (rate)"],
    "Employment Score": imd_df["Employment Score (rate)"],
    "Health Score":     imd_df["Health Deprivation and Disability Score"],
    "Living Env":       imd_df["Living Environment Score"],
    "Unemployment %":   (nomis_df["Economic activity status: Economically active (excluding full-time students): Unemployed"] /
                         nomis_df["Economic activity status: Economically active (excluding full-time students)"] * 100),
    "Population":       census_df["Residence type: Total; measures: Value"],
    "No-car HH %":      (cars_df["Number of cars or vans: No cars or vans in household"] /
                         cars_df["Number of cars or vans: Total: All households"] * 100),
}

print(f"  {'Metric':<22} {'Mean':>8} {'Std':>8} {'Skew':>7} {'Kurt':>7} {'IQR outliers':>13} {'Z>3.5':>7} {'Min':>8} {'Max':>8}")
print(f"  {'-'*105}")

outlier_summary = {}
for name, series in KEY_METRICS.items():
    s = series.dropna()
    mean_v = s.mean()
    std_v  = s.std()
    skew_v = float(scipy_stats.skew(s))
    kurt_v = float(scipy_stats.kurtosis(s))

    # IQR outliers
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_out = ((s < q1 - 3*iqr) | (s > q3 + 3*iqr)).sum()

    # Z-score outliers (|z| > 3.5)
    z_scores = np.abs((s - mean_v) / std_v)
    z_out    = (z_scores > 3.5).sum()

    print(f"  {name:<22} {mean_v:>8.2f} {std_v:>8.2f} {skew_v:>7.2f} {kurt_v:>7.2f} {iqr_out:>13,} {z_out:>7,} {s.min():>8.2f} {s.max():>8.2f}")

    sev = "LOW" if (iqr_out < N*0.01 and z_out < N*0.01) else "MEDIUM"
    log(f"S1.{name[:15]}.iqr_outliers", "Statistical", "Multiple", name, sev,
        "WARN" if iqr_out > 0 else "PASS",
        f"{iqr_out} IQR outliers", "<1% of rows", iqr_out, iqr_out/N*100,
        f"IQR outliers (3×IQR fence): skew={skew_v:.2f}, kurt={kurt_v:.2f}")

    outlier_summary[name] = {"iqr_outliers": int(iqr_out), "z_outliers": int(z_out),
                              "skew": round(skew_v,3), "kurtosis": round(kurt_v,3)}

# ── Isolation Forest on combined socio-economic features ─────────────────
print(f"\n  Isolation Forest — multivariate anomaly detection on 5 key metrics...")
feat_df = pd.DataFrame({
    "imd":      imd_df.set_index("LSOA code (2021)")["Index of Multiple Deprivation (IMD) Score"],
    "income":   imd_df.set_index("LSOA code (2021)")["Income Score (rate)"],
    "health":   imd_df.set_index("LSOA code (2021)")["Health Deprivation and Disability Score"],
    "unemp":    nomis_df.set_index("geography code")["Economic activity status: Economically active (excluding full-time students): Unemployed"] /
                nomis_df.set_index("geography code")["Economic activity status: Economically active (excluding full-time students)"] * 100,
    "pop":      census_df.set_index("geography code")["Residence type: Total; measures: Value"],
}).dropna()

iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
preds = iso.fit_predict(feat_df)
iso_anomalies = (preds == -1).sum()
anomaly_lsoas = feat_df.index[preds == -1].tolist()

log("S2.IsoForest.multivariate", "Statistical", "Multiple", "5 metrics", "MEDIUM",
    "WARN" if iso_anomalies > 0 else "PASS",
    f"{iso_anomalies} anomalies", f"<{int(N*0.01)} (1% contamination)",
    iso_anomalies, iso_anomalies/N*100,
    "Multivariate anomalies: LSOAs unusual across IMD, income, health, unemployment, population simultaneously")

print(f"  Isolation Forest flagged {iso_anomalies:,} LSOAs ({iso_anomalies/N*100:.1f}%) as multivariate outliers")
print(f"  Sample anomaly LSOAs: {anomaly_lsoas[:5]}")

GT["statistical_profiling"] = outlier_summary
GT["isolation_forest_anomalies"] = {"count": iso_anomalies, "sample": anomaly_lsoas[:10]}
print(f"\n  ✓ Statistical profiling complete")



  11d. Statistical Profiling — Skewness, Kurtosis, IQR Outliers, Z-Score, Isolation Forest
  Metric                     Mean      Std    Skew    Kurt  IQR outliers   Z>3.5      Min      Max
  ---------------------------------------------------------------------------------------------------------
  IMD Score                 21.67    15.69    1.14    0.93             4     104     0.17    94.22
  ⚠ 🟢 [S1.IMD Score.iqr_outliers] Multiple/IMD Score: IQR outliers (3×IQR fence): skew=1.14, kurt=0.93 (rows=4, 0.0%)
  Income Score               0.23     0.16    0.99    0.37             1      46     0.00     1.00
  ⚠ 🟢 [S1.Income Score.iqr_outliers] Multiple/Income Score: IQR outliers (3×IQR fence): skew=0.99, kurt=0.37 (rows=1, 0.0%)
  Employment Score           0.13     0.08    1.24    1.62            30     143     0.00     0.98
  ⚠ 🟢 [S1.Employment Scor.iqr_outliers] Multiple/Employment Score: IQR outliers (3×IQR fence): skew=1.24, kurt=1.62 (rows=30, 0.1%)
  Health Score               0

  ⚠ 🟡 [S2.IsoForest.multivariate] Multiple/5 metrics: Multivariate anomalies: LSOAs unusual across IMD, income, health, unemployment, population simultaneously (rows=338, 1.0%)
  Isolation Forest flagged 338 LSOAs (1.0%) as multivariate outliers
  Sample anomaly LSOAs: ['E01000601', 'E01001178', 'E01002039', 'E01004670', 'E01004794']

  ✓ Statistical profiling complete


In [28]:
section("11e. Cross-Dataset Consistency — Population Agreement Across All Sources")

# Build master comparison table: same LSOA, population from each source
master_pop = imd_df[["LSOA code (2021)","Total population: mid 2022"]].rename(
    columns={"LSOA code (2021)":"lsoa","Total population: mid 2022":"pop_imd_2022"})

census_pop = census_df[["geography code","Residence type: Total; measures: Value"]].rename(
    columns={"geography code":"lsoa","Residence type: Total; measures: Value":"pop_census_2021"})

age_pop = age_df[["geography code","Age: Total"]].rename(
    columns={"geography code":"lsoa","Age: Total":"pop_age"})

eth_pop = eth_df[["geography code","Ethnic group: Total: All usual residents"]].rename(
    columns={"geography code":"lsoa","Ethnic group: Total: All usual residents":"pop_eth"})

# Car ownership uses households not population — skip for population comparison
nomis_pop = nomis_df[["geography code",
    "Economic activity status: Total: All usual residents aged 16 years and over"]].rename(
    columns={"geography code":"lsoa",
             "Economic activity status: Total: All usual residents aged 16 years and over":"pop_16plus"})

# Merge all
comp = master_pop.merge(census_pop, on="lsoa", how="left")
comp = comp.merge(age_pop, on="lsoa", how="left")
comp = comp.merge(eth_pop, on="lsoa", how="left")
comp = comp.merge(nomis_pop, on="lsoa", how="left")

# Census vs Age: TS001=all residents, TS007a=usual residents — differences up to ~20 expected
# Only flag genuine errors (diff>50)
diff_age = (comp["pop_census_2021"] - comp["pop_age"]).abs()
bad_age  = (diff_age > 50).sum()
log("X1.Census_vs_Age.pop", "CrossDataset", "Census/Age", "Population",
    "MEDIUM" if bad_age>0 else "INFO",
    "WARN" if bad_age>0 else "PASS",
    f"{bad_age} LSOAs with diff>50", "0", bad_age, bad_age/N*100,
    f"Census (all residents TS001) vs Age (usual residents TS007a) mismatch >50. Max diff={diff_age.max()} — small diffs expected due to different reference populations")

# Census vs Ethnicity: should be within ±10
diff_eth = (comp["pop_census_2021"] - comp["pop_eth"]).abs()
bad_eth  = (diff_eth > 10).sum()
log("X2.Census_vs_Eth.pop", "CrossDataset", "Census/Ethnicity", "Population",
    "MEDIUM" if bad_eth>0 else "INFO",
    "WARN" if bad_eth>0 else "PASS",
    f"{bad_eth} LSOAs with diff>10", "0", bad_eth, bad_eth/N*100,
    f"Census pop vs Ethnicity total mismatch >10. Max diff={diff_eth.max()}")

# Print distribution of differences
print(f"  Population consistency across datasets:")
print(f"  Census vs Age    — mean diff: {diff_age.mean():.2f}, max diff: {diff_age.max():.0f}, LSOAs >5: {bad_age}")
print(f"  Census vs Eth    — mean diff: {diff_eth.mean():.2f}, max diff: {diff_eth.max():.0f}, LSOAs >10: {bad_eth}")
print(f"  IMD 2022 est vs Census 2021 — mean diff: {(comp['pop_imd_2022']-comp['pop_census_2021']).abs().mean():.1f} (expected: ~50-100, different years)")

# Worst mismatches
if bad_age > 0:
    worst = comp.nlargest(5, "diff_age" if "diff_age" in comp else "pop_census_2021")
    bad_rows = comp[diff_age > 5][["lsoa","pop_census_2021","pop_age"]].head(5)
    print(f"\n  Worst Census vs Age mismatches:")
    print(bad_rows.to_string(index=False))

if bad_eth > 0:
    bad_eth_rows = comp[diff_eth > 10][["lsoa","pop_census_2021","pop_eth"]].head(5)
    print(f"\n  Worst Census vs Ethnicity mismatches:")
    print(bad_eth_rows.to_string(index=False))

GT["cross_dataset_consistency"] = {
    "census_vs_age_mismatches_gt5":  int(bad_age),
    "census_vs_eth_mismatches_gt10": int(bad_eth),
    "census_vs_age_max_diff":        int(diff_age.max()),
    "census_vs_eth_max_diff":        int(diff_eth.max()),
}
print(f"\n  ✓ Cross-dataset consistency complete")



  11e. Cross-Dataset Consistency — Population Agreement Across All Sources
  Population consistency across datasets:
  Census vs Age    — mean diff: 1.76, max diff: 13, LSOAs >5: 0
  Census vs Eth    — mean diff: 1.45, max diff: 10, LSOAs >10: 0
  IMD 2022 est vs Census 2021 — mean diff: 40.0 (expected: ~50-100, different years)

  ✓ Cross-dataset consistency complete


In [29]:
section("11f. Correlation Analysis — Are Inter-Variable Relationships Physically Sensible?")

# Build combined frame with short column aliases for readability
imd_sub = imd_df[["LSOA code (2021)",
    "Index of Multiple Deprivation (IMD) Score",
    "Income Score (rate)",
    "Employment Score (rate)",
    "Health Deprivation and Disability Score",
    "Living Environment Score",
    "Crime Score",
]].rename(columns={
    "LSOA code (2021)":                                    "lsoa",
    "Index of Multiple Deprivation (IMD) Score":           "imd_score",
    "Income Score (rate)":                                 "income_score",
    "Employment Score (rate)":                             "employ_score",
    "Health Deprivation and Disability Score":             "health_score",
    "Living Environment Score":                            "living_score",
    "Crime Score":                                         "crime_score",
})

nomis_sub = nomis_df[["geography code",
    "Economic activity status: Economically active (excluding full-time students): Unemployed",
    "Economic activity status: Economically active (excluding full-time students)",
]].rename(columns={
    "geography code": "lsoa",
    "Economic activity status: Economically active (excluding full-time students): Unemployed": "unemployed",
    "Economic activity status: Economically active (excluding full-time students)": "econ_active",
})

census_sub = census_df[["geography code","Residence type: Total; measures: Value"]].rename(
    columns={"geography code":"lsoa","Residence type: Total; measures: Value":"population"})

cars_sub = cars_df[["geography code",
    "Number of cars or vans: Total: All households",
    "Number of cars or vans: No cars or vans in household",
]].rename(columns={
    "geography code":"lsoa",
    "Number of cars or vans: Total: All households":"total_hh",
    "Number of cars or vans: No cars or vans in household":"no_car_hh",
})

combined = imd_sub.merge(nomis_sub, on="lsoa")
combined = combined.merge(census_sub, on="lsoa")
combined = combined.merge(cars_sub, on="lsoa")
combined["unemp_rate"]  = combined["unemployed"] / combined["econ_active"] * 100
combined["nocar_pct"]   = combined["no_car_hh"] / combined["total_hh"] * 100

# Expected correlations (domain knowledge)
# Format: (col_a, col_b, direction_label, minimum_expected_r)
expected = [
    ("imd_score",    "income_score",  "STRONG POSITIVE",    0.85),
    ("imd_score",    "employ_score",  "STRONG POSITIVE",    0.80),
    ("imd_score",    "health_score",  "STRONG POSITIVE",    0.70),
    ("imd_score",    "unemp_rate",    "STRONG POSITIVE",    0.55),
    ("imd_score",    "nocar_pct",     "MODERATE POSITIVE",  0.30),
    ("income_score", "unemp_rate",    "STRONG POSITIVE",    0.65),
    ("imd_score",    "crime_score",   "MODERATE POSITIVE",  0.40),
    ("imd_score",    "living_score",  "MODERATE",          -0.99),  # can be pos or neg
]

print(f"  {'Pair':<45} {'Expected':>18} {'Observed':>10} {'Status'}")
print(f"  {'-'*85}")

for col_a, col_b, direction, min_r in expected:
    valid = combined[[col_a, col_b]].dropna()
    r, p = scipy_stats.pearsonr(valid[col_a], valid[col_b])
    if direction == "MODERATE":
        ok = True  # Living Env can go either way
    elif "POSITIVE" in direction:
        ok = r >= min_r
    else:
        ok = r <= -abs(min_r)
    status = "✓" if ok else "⚠ BELOW EXPECTED"
    pair = f"{col_a} × {col_b}"
    print(f"  {status}  {pair:<45} r={r:>6.3f}  (expected {direction} >={min_r})")
    log(f"R1.{col_a}_x_{col_b}", "Correlation", "Multiple", pair,
        "MEDIUM" if not ok else "INFO",
        "WARN" if not ok else "PASS",
        f"r={r:.3f}", f">={min_r} {direction}", 0, 0,
        f"Pearson r={r:.3f} vs expected {direction} (min r={min_r})")

# Full correlation matrix
corr_cols = ["imd_score","income_score","employ_score","health_score",
             "crime_score","unemp_rate","nocar_pct","population"]
corr_matrix = combined[corr_cols].corr().round(3)
print(f"\n  Full correlation matrix:")
print(corr_matrix.to_string())

GT["correlation_analysis"] = {
    f"{a}_x_{b}": round(float(scipy_stats.pearsonr(
        combined[[a,b]].dropna()[a], combined[[a,b]].dropna()[b])[0]), 3)
    for a, b, _, _ in expected
}
print(f"\n  ✓ Correlation analysis complete")



  11f. Correlation Analysis — Are Inter-Variable Relationships Physically Sensible?
  Pair                                                    Expected   Observed Status
  -------------------------------------------------------------------------------------
  ✓  imd_score × income_score                      r= 0.954  (expected STRONG POSITIVE >=0.85)
  ✓  imd_score × employ_score                      r= 0.957  (expected STRONG POSITIVE >=0.8)
  ✓  imd_score × health_score                      r= 0.842  (expected STRONG POSITIVE >=0.7)
  ✓  imd_score × unemp_rate                        r= 0.792  (expected STRONG POSITIVE >=0.55)
  ✓  imd_score × nocar_pct                         r= 0.644  (expected MODERATE POSITIVE >=0.3)
  ✓  income_score × unemp_rate                     r= 0.807  (expected STRONG POSITIVE >=0.65)
  ✓  imd_score × crime_score                       r= 0.797  (expected MODERATE POSITIVE >=0.4)
  ✓  imd_score × living_score                      r= 0.288  (expected MODERA

In [30]:
section("11g. Benford's Law — Leading Digit Analysis on Count Columns")

# Benford expected distribution
BENFORD = {d: np.log10(1 + 1/d) for d in range(1,10)}

def benfords_test(series, name, ds_name):
    """Chi-squared test for Benford conformance."""
    s = series.dropna().astype(int)
    s = s[s > 0]  # only positive integers
    if len(s) < 100:
        return None
    leading = s.astype(str).str[0].astype(int)
    observed_pct = leading.value_counts(normalize=True).reindex(range(1,10), fill_value=0)
    expected_pct = pd.Series(BENFORD)
    # Chi-squared
    obs_counts = observed_pct * len(s)
    exp_counts = expected_pct * len(s)
    chi2, p = scipy_stats.chisquare(obs_counts, exp_counts)
    mad = abs(observed_pct - expected_pct).mean()  # mean absolute deviation
    conforms = mad < 0.015  # MAD < 1.5% = close conformance
    return {"chi2": round(chi2,2), "p": round(p,4), "mad": round(mad,4), "conforms": conforms, "n": len(s)}

print(f"  {'Dataset/Column':<45} {'N':>7} {'MAD':>7} {'chi2':>8} {'p-val':>8} {'Benford?'}")
print(f"  {'-'*85}")

benford_cols = [
    (census_df, "geography code", "Residence type: Total; measures: Value", "Census/Population"),
    (nomis_df,  "geography code", "Economic activity status: Total: All usual residents aged 16 years and over", "NOMIS/Pop16+"),
    (nomis_df,  "geography code", "Economic activity status: Economically active (excluding full-time students): Unemployed", "NOMIS/Unemployed"),
    (age_df,    "geography code", "Age: Total", "Age/Total"),
    (cars_df,   "geography code", "Number of cars or vans: Total: All households", "Cars/Households"),
    (imd_df,    "LSOA code (2021)", "Total population: mid 2022", "IMD/Pop2022"),
]

for df, key, col, label in benford_cols:
    result = benfords_test(df[col], col, label)
    if result:
        icon = "✓ Yes" if result["conforms"] else "⚠ No"
        print(f"  {label:<45} {result['n']:>7,} {result['mad']:>7.4f} {result['chi2']:>8.1f} {result['p']:>8.4f} {icon}")
        if not result["conforms"]:
            log(f"B1.{label[:20]}.benford", "Benford", label.split("/")[0], col,
                "LOW", "WARN",
                f"MAD={result['mad']:.4f}", "MAD<0.015",
                0, 0, f"Possible data anomaly: count column does not follow Benford's Law")

print(f"\n  Note: Census counts naturally follow Benford (aggregated from individual records).")
print(f"  Non-conformance at LSOA level may indicate capping, rounding, or synthetic data.")
print(f"\n  ✓ Benford's Law analysis complete")



  11g. Benford's Law — Leading Digit Analysis on Count Columns
  Dataset/Column                                      N     MAD     chi2    p-val Benford?
  -------------------------------------------------------------------------------------
  Census/Population                              33,755  0.1223  51648.8   0.0000 ⚠ No
  ⚠ 🟢 [B1.Census/Population.benford] Census/Residence type: Total; measures: Value: Possible data anomaly: count column does not follow Benford's Law (rows=0, 0.0%)
  NOMIS/Pop16+                                   33,755  0.1384  63102.6   0.0000 ⚠ No
  ⚠ 🟢 [B1.NOMIS/Pop16+.benford] NOMIS/Economic activity status: Total: All usual residents aged 16 years and over: Possible data anomaly: count column does not follow Benford's Law (rows=0, 0.0%)
  NOMIS/Unemployed                               33,754  0.0472   6907.4   0.0000 ⚠ No
  ⚠ 🟢 [B1.NOMIS/Unemployed.benford] NOMIS/Economic activity status: Economically active (excluding full-time students): Unemployed: Pos

In [31]:
section("11h. Business Rule Validation — Domain-Specific Rules")

# Rule 1: IMD decile band sizes should be approximately equal (±5% of N/10)
decile_counts = imd_df["Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"].value_counts()
expected_per_decile = N / 10
for d in range(1,11):
    count = decile_counts.get(d, 0)
    deviation_pct = abs(count - expected_per_decile) / expected_per_decile * 100
    bad = deviation_pct > 5
    log(f"BR1.IMD.decile_{d}_size", "BusinessRule", "IMD2025", f"Decile {d}",
        "MEDIUM" if bad else "INFO",
        "WARN" if bad else "PASS",
        count, f"≈{int(expected_per_decile)}±5%",
        0, 0, f"Decile {d} has {count} LSOAs ({deviation_pct:.1f}% deviation from expected)")

print(f"  Decile band sizes: {dict(decile_counts.sort_index())}")

# Rule 2: Unemployment rate at national level should be ~5% (ONS published: 4.8% Census 2021)
econ_active = nomis_df["Economic activity status: Economically active (excluding full-time students)"]
unemployed  = nomis_df["Economic activity status: Economically active (excluding full-time students): Unemployed"]
national_unemp = unemployed.sum() / econ_active.sum() * 100
log("BR2.NOMIS.national_rate", "BusinessRule", "NOMIS", "Unemployment rate",
    "HIGH" if abs(national_unemp - 4.8) > 1.0 else "INFO",
    "WARN" if abs(national_unemp - 4.8) > 1.0 else "PASS",
    f"{national_unemp:.2f}%", "~4.8% (ONS Census 2021)",
    0, 0, f"National unemployment rate: {national_unemp:.2f}% vs ONS published 4.8%")
print(f"\n  National unemployment rate (computed): {national_unemp:.3f}%  (ONS published: ~4.8%)")

# Rule 3: England total population should be ~56.5M (ONS Census 2021: 56,490,048)
total_pop = census_df["Residence type: Total; measures: Value"].sum()
pop_diff_pct = abs(total_pop - 56490048) / 56490048 * 100
log("BR3.Census.total_pop", "BusinessRule", "Census", "Total population",
    "HIGH" if pop_diff_pct > 0.1 else "INFO",
    "WARN" if pop_diff_pct > 0.1 else "PASS",
    f"{total_pop:,}", "56,490,048 (ONS 2021)",
    0, 0, f"England total population: {total_pop:,} vs ONS {56490048:,} (diff={pop_diff_pct:.4f}%)")
print(f"  England total population: {total_pop:,}  (ONS published: 56,490,048  diff={pop_diff_pct:.4f}%)")

# Rule 4: No LSOA should have population < 100 (ONS minimum is ~1,000)
tiny_lsoas = (census_df["Residence type: Total; measures: Value"] < 100).sum()
log("BR4.Census.min_pop", "BusinessRule", "Census", "Population",
    "HIGH" if tiny_lsoas>0 else "INFO",
    "FAIL" if tiny_lsoas>0 else "PASS",
    tiny_lsoas, ">100 expected", tiny_lsoas, tiny_lsoas/N*100,
    "LSOAs with population < 100 (below ONS minimum design threshold)")

# Rule 5: No LSOA should have population > 10,000 (ONS max design: ~3,000 but extremes exist)
giant_lsoas = (census_df["Residence type: Total; measures: Value"] > 10000).sum()
log("BR5.Census.max_pop", "BusinessRule", "Census", "Population",
    "MEDIUM" if giant_lsoas>0 else "INFO",
    "WARN" if giant_lsoas>0 else "PASS",
    giant_lsoas, "<10,000 typical", giant_lsoas, giant_lsoas/N*100,
    "LSOAs with population > 10,000 — far above ONS design maximum of ~3,000")
print(f"  LSOAs with population > 10,000: {giant_lsoas}")
print(f"  Min LSOA population: {census_df['Residence type: Total; measures: Value'].min()}")
print(f"  Max LSOA population: {census_df['Residence type: Total; measures: Value'].max()}")

# Rule 6: All LSOAs in RUC should have a valid flag
missing_ruc_flag = ruc_df["Urban_rural_flag"].isna().sum()
log("BR6.RUC.flag_complete", "BusinessRule", "RUC", "Urban_rural_flag",
    "HIGH" if missing_ruc_flag>0 else "INFO",
    "FAIL" if missing_ruc_flag>0 else "PASS",
    missing_ruc_flag, 0, missing_ruc_flag, missing_ruc_flag/N*100,
    "LSOAs with missing urban/rural flag")

# Rule 7: Urban LSOAs should have higher mean stop counts than rural (from section 9)
print(f"\n  ✓ Business rule validation complete")



  11h. Business Rule Validation — Domain-Specific Rules
  Decile band sizes: {1: np.int64(3375), 2: np.int64(3376), 3: np.int64(3375), 4: np.int64(3376), 5: np.int64(3375), 6: np.int64(3376), 7: np.int64(3375), 8: np.int64(3376), 9: np.int64(3375), 10: np.int64(3376)}

  National unemployment rate (computed): 4.871%  (ONS published: ~4.8%)
  England total population: 56,490,056  (ONS published: 56,490,048  diff=0.0000%)
  LSOAs with population > 10,000: 0
  Min LSOA population: 999
  Max LSOA population: 9899

  ✓ Business rule validation complete


In [32]:
section("11i. Accuracy Cross-Check — Verify Against ONS Published Aggregates")

print("  Checking computed aggregates against ONS/DLUHC published figures:\n")

checks = []

# 1. England population
pop_computed = int(census_df["Residence type: Total; measures: Value"].sum())
pop_ons      = 56490048
checks.append(("England population", pop_computed, pop_ons, 56490000, 56500000))

# 2. Number of England LSOAs
n_lsoas = len(imd_df)
checks.append(("England LSOA count", n_lsoas, 33755, 33750, 33760))

# 3. Number of IMD deciles (must be exactly 10)
n_deciles = imd_df["Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"].nunique()
checks.append(("IMD decile count", n_deciles, 10, 10, 10))

# 4. National unemployment rate (Census 2021: 4.8% per ONS)
econ_act = nomis_df["Economic activity status: Economically active (excluding full-time students)"].sum()
unemp_ct = nomis_df["Economic activity status: Economically active (excluding full-time students): Unemployed"].sum()
unemp_rate_nat = round(unemp_ct / econ_act * 100, 2)
checks.append(("National unemp rate %", unemp_rate_nat, 4.8, 4.0, 6.0))

# 5. % urban LSOAs (ONS RUC 2021: ~83% urban)
pct_urban = round(ruc_df["Urban_rural_flag"].eq("Urban").mean() * 100, 1)
checks.append(("% Urban LSOAs", pct_urban, 82.6, 80.0, 86.0))

# 6. RUC UN1 (core urban) should be ~75-80% of all LSOAs
pct_un1 = round((ruc_df["RUC21CD"] == "UN1").mean() * 100, 1)
checks.append(("% UN1 (core urban) LSOAs", pct_un1, 75.9, 72.0, 79.0))

print(f"  {'Check':<35} {'Computed':>12} {'Reference':>12} {'Range':>20} {'Status'}")
print(f"  {'-'*90}")

accuracy_results = {}
for name, computed, reference, lo, hi in checks:
    ok = lo <= computed <= hi
    status = "✓ PASS" if ok else "⚠ WARN"
    print(f"  {status}  {name:<35} {str(computed):>12} {str(reference):>12} [{lo}–{hi}]")
    log(f"A1.{name[:20].replace(' ','_')}", "Accuracy", "Multiple", name,
        "HIGH" if not ok else "INFO",
        "FAIL" if not ok else "PASS",
        str(computed), f"[{lo},{hi}]", 0, 0,
        f"Accuracy check vs published aggregate: computed={computed}, reference={reference}")
    accuracy_results[name] = {"computed": computed, "reference": reference, "pass": ok}

GT["accuracy_checks"] = accuracy_results
print(f"\n  ✓ Accuracy cross-check complete")



  11i. Accuracy Cross-Check — Verify Against ONS Published Aggregates
  Checking computed aggregates against ONS/DLUHC published figures:

  Check                                   Computed    Reference                Range Status
  ------------------------------------------------------------------------------------------
  ✓ PASS  England population                      56490056     56490048 [56490000–56500000]
  ✓ PASS  England LSOA count                         33755        33755 [33750–33760]
  ✓ PASS  IMD decile count                              10           10 [10–10]
  ✓ PASS  National unemp rate %                       4.87          4.8 [4.0–6.0]
  ✓ PASS  % Urban LSOAs                               83.5         82.6 [80.0–86.0]
  ✓ PASS  % UN1 (core urban) LSOAs                    77.6         75.9 [72.0–79.0]

  ✓ Accuracy cross-check complete


In [33]:
section("11j. Cardinality Analysis + Encoding/Structural Anomalies")

print("  Cardinality Analysis — suspiciously low or high unique counts:\n")

cardinality_checks = [
    (imd_df,    "LSOA code (2021)",             N,     N,   "Should be exactly N (unique LSOA codes)"),
    (imd_df,    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)", N, N, "Should be exactly N (bijection)"),
    (imd_df,    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)", 10, 10, "Should be exactly 10"),
    (ruc_df,    "RUC21CD",                       6,     6,   "Should be exactly 6 RUC categories"),
    (ruc_df,    "Urban_rural_flag",               2,     2,   "Should be exactly 2 (Urban/Rural)"),
    (nomis_df,  "date",                           1,     1,   "Should be exactly 1 (2021 only)"),
    (naptan_df, "StopType",                      18,    25,   "NaPTAN has ~18-20 stop types"),
    (naptan_df, "Status",                         3,     3,   "Should be active/inactive/pending"),
]

print(f"  {'Dataset/Column':<55} {'Unique':>8} {'Expected':>10} {'Status'}")
print(f"  {'-'*80}")

for df, col, exp_lo, exp_hi, note in cardinality_checks:
    n_unique = df[col].nunique()
    ok = exp_lo <= n_unique <= exp_hi
    status = "✓" if ok else "⚠"
    print(f"  {status} {col[:55]:<55} {n_unique:>8,} {f'[{exp_lo},{exp_hi}]':>10}  {note}")
    if not ok:
        log(f"K1.{col[:20]}.cardinality", "Cardinality", "Multiple", col,
            "HIGH", "WARN",
            f"{n_unique} unique", f"[{exp_lo},{exp_hi}]", 0, 0,
            f"Unexpected cardinality: {note}")

# ── Encoding/Structural anomalies ────────────────────────────────────────
print(f"\n  Structural Anomaly Checks:")

# Check for numeric columns stored as strings (type coercion issues)
for ds_name, (df, key) in DATASETS.items():
    for col in df.columns:
        if df[col].dtype == object:
            # Try to see if it should be numeric
            sample = df[col].dropna().head(100)
            try:
                pd.to_numeric(sample)
                # It converted — might be a string-stored number
                non_numeric = pd.to_numeric(df[col], errors="coerce").isna().sum() - df[col].isna().sum()
                if non_numeric == 0 and df[col].notna().sum() > 100:
                    log(f"E1.{ds_name}.{col[:20]}_str_num", "Encoding", ds_name, col,
                        "LOW", "WARN",
                        "string", "numeric", 0, 0,
                        f"Numeric data stored as string type — will cause silent type errors in arithmetic")
            except:
                pass

# Check for mixed types in key columns (LSOA codes should ALL be string E+digits)
for ds_name, (df, key) in DATASETS.items():
    mixed = df[key].apply(type).nunique()
    if mixed > 1:
        log(f"E2.{ds_name}.{key}_mixed_types", "Encoding", ds_name, key,
            "CRITICAL", "FAIL", f"{mixed} types", 1,
            (df[key].apply(type) != str).sum(), 0,
            "Mixed types in join key column")

# Check for BOM / invisible characters in LSOA codes
for ds_name, (df, key) in DATASETS.items():
    bom_count = df[key].astype(str).str.contains("\ufeff|\u200b|\xa0", regex=True, na=False).sum()
    if bom_count > 0:
        log(f"E3.{ds_name}.{key}_bom", "Encoding", ds_name, key,
            "CRITICAL", "FAIL", f"{bom_count} rows with BOM/invisible chars", 0,
            bom_count, bom_count/len(df)*100, "BOM or invisible characters in join key")

print(f"  ✓ Cardinality and encoding checks complete")



  11j. Cardinality Analysis + Encoding/Structural Anomalies
  Cardinality Analysis — suspiciously low or high unique counts:

  Dataset/Column                                            Unique   Expected Status
  --------------------------------------------------------------------------------
  ✓ LSOA code (2021)                                          33,755 [33755,33755]  Should be exactly N (unique LSOA codes)
  ✓ Index of Multiple Deprivation (IMD) Rank (where 1 is mo   33,755 [33755,33755]  Should be exactly N (bijection)
  ✓ Index of Multiple Deprivation (IMD) Decile (where 1 is        10    [10,10]  Should be exactly 10
  ✓ RUC21CD                                                        6      [6,6]  Should be exactly 6 RUC categories
  ✓ Urban_rural_flag                                               2      [2,2]  Should be exactly 2 (Urban/Rural)
  ✓ date                                                           1      [1,1]  Should be exactly 1 (2021 only)
  ✓ StopType       

  ✓ Cardinality and encoding checks complete


In [34]:
section("11k. Audit Summary — Full Report")

audit_df = pd.DataFrame(AUDIT_LOG)

print(f"  Total checks run: {len(audit_df):,}")
print()

# By status
status_counts = audit_df["status"].value_counts()
for status, count in status_counts.items():
    icon = {"PASS":"✓","FAIL":"✗","WARN":"⚠"}.get(status,"·")
    print(f"  {icon} {status}: {count}")

print()

# By severity (non-PASS only)
issues = audit_df[audit_df["status"] != "PASS"]
if len(issues) > 0:
    print(f"  Issues by severity:")
    sev_counts = issues["severity"].value_counts()
    for sev, count in sev_counts.items():
        icon = {"CRITICAL":"🔴","HIGH":"🟠","MEDIUM":"🟡","LOW":"🟢"}.get(sev,"⚪")
        print(f"    {icon} {sev}: {count}")

    print(f"\n  CRITICAL and HIGH issues requiring action:")
    blockers = issues[issues["severity"].isin(["CRITICAL","HIGH"])]
    if len(blockers) == 0:
        print(f"  ✓ No CRITICAL or HIGH issues found")
    else:
        for _, row in blockers.iterrows():
            print(f"  ✗ [{row['check_id']}] {row['dataset']}/{row['column']}")
            print(f"      {row['notes']}")
            print(f"      Observed: {row['observed']} | Expected: {row['expected']} | Rows: {row['affected_rows']}")
else:
    print(f"  ✓ No issues found")

# Save audit log
audit_csv = AUDIT_DIR / "data_quality_audit_log.csv"
audit_df.to_csv(audit_csv, index=False)
print(f"\n  Full audit log saved: {audit_csv}")
print(f"  Rows: {len(audit_df):,} checks")

GT["data_quality_audit"] = {
    "total_checks":     len(audit_df),
    "pass":             int(status_counts.get("PASS", 0)),
    "fail":             int(status_counts.get("FAIL", 0)),
    "warn":             int(status_counts.get("WARN", 0)),
    "critical_issues":  int((issues["severity"] == "CRITICAL").sum()) if len(issues) > 0 else 0,
    "high_issues":      int((issues["severity"] == "HIGH").sum()) if len(issues) > 0 else 0,
    "audit_log_path":   "data/audit/data_quality_audit_log.csv",
}
print(f"\n  ✓ Exhaustive data quality investigation complete")
if GT["data_quality_audit"]["critical_issues"] == 0 and GT["data_quality_audit"]["high_issues"] == 0:
    print(f"  ✓ PIPELINE CLEAR — no blocking data quality issues found")
else:
    print(f"  ⚠ PIPELINE BLOCKED — resolve CRITICAL/HIGH issues before Phase 1")



  11k. Audit Summary — Full Report
  Total checks run: 103

  ✓ PASS: 89
  ⚠ WARN: 14

  Issues by severity:
    🟢 LOW: 11
    🟡 MEDIUM: 2
    🟠 HIGH: 1

  CRITICAL and HIGH issues requiring action:
  ✗ [C6.NaPTAN.coords] NaPTAN/Latitude
      Null coordinates in raw NaPTAN (52,449 rows — filtered out in England set)
      Observed: 52449 | Expected: 0 | Rows: 52449

  Full audit log saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/data_quality_audit_log.csv
  Rows: 103 checks

  ✓ Exhaustive data quality investigation complete
  ⚠ PIPELINE BLOCKED — resolve CRITICAL/HIGH issues before Phase 1


# 8. Lock Ground Truth

Write `ground_truth.json`. This file is the contract for the entire pipeline.  
**Do not edit this file manually after locking. If numbers change, re-run the audit.**

In [35]:
section("8. Lock Ground Truth")

import numpy as np

class _NpEncoder(json_lib.JSONEncoder):
    """Encode numpy int/float/bool as native Python types."""
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_): return bool(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

GT_PATH = AUDIT_DIR / "ground_truth.json"
GT["locked_at"] = datetime.utcnow().isoformat()

with open(GT_PATH, "w") as f:
    json_lib.dump(GT, f, indent=2, cls=_NpEncoder)

print(f"  Ground truth written to: {GT_PATH}")
print(f"\n  Summary:")

def _fmt(val, label):
    if isinstance(val, (int, float)):
        return f"    {label}: {val:,}"
    return f"    {label}: {val or 'NOT SET'}"

print(_fmt(GT['naptan'].get('england_active_bus_stops'), "NaPTAN England active bus stops"))
print(_fmt(GT['bods'].get('unique_bus_route_ids'), "BODS unique bus routes"))
print(_fmt(GT['census'].get('total_lsoas_england'), "Census 2021 England LSOAs"))
print(_fmt(GT['census'].get('total_population_sum'), "Census population sum"))
print(_fmt(GT['imd'].get('total_lsoas'), "IMD 2025 LSOAs"))
print(_fmt(GT['imd'].get('lsoas_no_imd_score'), "LSOAs with no IMD score"))
print(_fmt(GT['nomis'].get('total_lsoas_england'), "NOMIS TS066 LSOAs"))
print(f"    Stop→LSOA match rate: {GT['joins'].get('stop_to_lsoa_match_rate_pct', 'NOT SET')}%")

dqa = GT.get('data_quality_audit', {})
print(f"\n  Data Quality Audit:")
print(f"    Total checks:    {dqa.get('total_checks', 'NOT RUN')}")
print(f"    PASS:            {dqa.get('pass', '-')}")
print(f"    FAIL/WARN:       {dqa.get('fail', 0) + dqa.get('warn', 0)}")
print(f"    CRITICAL issues: {dqa.get('critical_issues', '-')}")
print(f"    HIGH issues:     {dqa.get('high_issues', '-')}")

print(f"\n  ✓ Audit complete. Ground truth locked.")
print(f"  Next step: build src/aequitas/ package starting with ingestion layer.")



  8. Lock Ground Truth
  Ground truth written to: /Users/souravamseekarmarti/Projects/aequitas/data/audit/ground_truth.json

  Summary:
    NaPTAN England active bus stops: 274,719
    BODS unique bus routes: 13,099
    Census 2021 England LSOAs: 33,755
    Census population sum: 56,490,056
    IMD 2025 LSOAs: 33,755
    LSOAs with no IMD score: 0
    NOMIS TS066 LSOAs: 33,755
    Stop→LSOA match rate: 99.9993%

  Data Quality Audit:
    Total checks:    103
    PASS:            89
    FAIL/WARN:       14
    CRITICAL issues: 0
    HIGH issues:     1

  ✓ Audit complete. Ground truth locked.
  Next step: build src/aequitas/ package starting with ingestion layer.
